# Select small set

In [5]:
from collections import Counter
from datasets import concatenate_datasets

TARGET_SIZE = 120_000
BALANCED_TASKS = ("GSM8K", "MATH")
SEED = 123

def _normalize_task(name: str) -> str:
    return name.strip().lower()

def select_balanced_subset(dataset, tasks=BALANCED_TASKS, target_size=TARGET_SIZE, seed=SEED):
    normalized_tasks = [_normalize_task(task) for task in tasks]
    per_task, remainder = divmod(target_size, len(normalized_tasks))
    allocations = {task: per_task for task in normalized_tasks}
    for index in range(remainder):
        allocations[normalized_tasks[index]] += 1

    subsets = []
    actual_allocations = {}

    for raw_task, normalized_task in zip(tasks, normalized_tasks):
        task_ds = dataset.filter(
            lambda example, normalized_task=normalized_task: _normalize_task(example["task"]) == normalized_task
        )
        available = len(task_ds)
        take = min(allocations[normalized_task], available)
        if take < allocations[normalized_task]:
            print(f"Warning: Only {available} examples available for task {raw_task}. Taking all of them.")
        subset = task_ds.shuffle(seed=seed).select(range(take))
        actual_allocations[raw_task] = len(subset)
        subsets.append(subset)

    combined = concatenate_datasets(subsets).shuffle(seed=seed)
    print("Selected counts per task:")
    for task in tasks:
        print(f"  {task}: {actual_allocations.get(task, 0)}")
    print(f"Total selected: {len(combined)} (target was {target_size})")
    return combined

def analyze_value_distribution(dataset):
    counts = Counter()
    for values in dataset["value"]:
        counts.update(values)

    total = sum(counts.values())
    print("Value token distribution:")
    for label in ["+", "-"]:
        portion = counts[label] / total if total else 0.0
        print(f"  {label}: {counts[label]} ({portion:.2%})")

    other_keys = [key for key in counts.keys() if key not in {"+", "-"}]
    if other_keys:
        print("Other tokens:")
        for key in other_keys:
            portion = counts[key] / total if total else 0.0
            print(f"  {key}: {counts[key]} ({portion:.2%})")

    return counts


In [ ]:
from datasets import load_dataset
ds = load_dataset("zhuzilin/Math-Shepherd")
small_train = select_balanced_subset(ds["train"])
value_counts = analyze_value_distribution(small_train)

from datasets import DatasetDict, load_from_disk
OUTPUT_DIR = "/home/leena/ccc_eval/rs_prm/data/math_shepherd_small_120k"
DatasetDict({"train": small_train}).save_to_disk(OUTPUT_DIR)
small_ds = load_from_disk(OUTPUT_DIR)["train"]
print(len(small_ds))

Selected counts per task:
  GSM8K: 60000
  MATH: 60000
Total selected: 120000 (target was 120000)
Value token distribution:
  +: 335378 (48.63%)
  -: 354204 (51.37%)


## ver2

In [1]:
from collections import Counter
from datasets import concatenate_datasets, load_dataset, Dataset, load_from_disk
import random

TARGET_SIZE = 120_000
BALANCED_TASKS = ("GSM8K", "MATH")
SEED = 777

def _normalize_task(name: str) -> str:
    return name.strip().lower()

def _count_pm(values):
    """values: list of '+' / '-' (기타 토큰은 무시)"""
    c = Counter(v for v in values if v in {"+", "-"})
    return int(c.get("+", 0)), int(c.get("-", 0))

def _prepare_task_ds(ds: Dataset, task_norm: str, seed: int):
    # task 필터
    tds = ds.filter(lambda ex, task_norm=task_norm: _normalize_task(ex["task"]) == task_norm)
    # 각 예제의 +, - 개수 계산
    tds = tds.map(
        lambda ex: {"plus_cnt": _count_pm(ex["value"])[0], "minus_cnt": _count_pm(ex["value"])[1]},
        desc=f"count +/- for {task_norm}"
    )
    # 셔플(재현성)
    tds = tds.shuffle(seed=seed)
    return tds

def _greedy_balance_pick(tds: Dataset, k: int, seed: int):
    """현재 누적 +/- 차이를 줄이는 방향으로 k개 고르기 (불가능하면 채울 때까지 보충)."""
    # 분기: +가 많은 샘플 / -가 많은 샘플 / 동률
    pos_dominant_idx = [i for i, ex in enumerate(tds) if ex["plus_cnt"] > ex["minus_cnt"]]
    neg_dominant_idx = [i for i, ex in enumerate(tds) if ex["minus_cnt"] > ex["plus_cnt"]]
    tie_idx          = [i for i, ex in enumerate(tds) if ex["minus_cnt"] == ex["plus_cnt"]]

    rng = random.Random(seed)
    rng.shuffle(pos_dominant_idx)
    rng.shuffle(neg_dominant_idx)
    rng.shuffle(tie_idx)

    sel_idx = []
    total_plus = 0
    total_minus = 0

    def take_from(bucket):
        if not bucket: 
            return False
        i = bucket.pop()  # 뒤에서 pop (셔플되어 있음)
        sel_idx.append(i)
        nonlocal total_plus, total_minus
        p, m = tds[i]["plus_cnt"], tds[i]["minus_cnt"]
        total_plus  += p
        total_minus += m
        return True

    while len(sel_idx) < k and (pos_dominant_idx or neg_dominant_idx or tie_idx):
        # 현재 어떤 쪽이 부족한지 보고 선택
        if total_plus <= total_minus:
            # +를 늘리는 쪽을 우선
            if not take_from(pos_dominant_idx):
                if not take_from(tie_idx):
                    take_from(neg_dominant_idx)
        else:
            # -를 늘리는 쪽을 우선
            if not take_from(neg_dominant_idx):
                if not take_from(tie_idx):
                    take_from(pos_dominant_idx)

    # 남았는데도 부족하면(모든 버킷 고갈) 그냥 앞에서 채움
    if len(sel_idx) < k:
        remaining = [i for i in range(len(tds)) if i not in set(sel_idx)]
        rng.shuffle(remaining)
        sel_idx.extend(remaining[: (k - len(sel_idx))])

    sel = tds.select(sel_idx)
    return sel

def select_balanced_subset(dataset, tasks=BALANCED_TASKS, target_size=TARGET_SIZE, seed=SEED, balance_value=True):
    normalized_tasks = [_normalize_task(t) for t in tasks]
    per_task, remainder = divmod(target_size, len(normalized_tasks))
    allocations = {t: per_task for t in normalized_tasks}
    for i in range(remainder):
        allocations[normalized_tasks[i]] += 1

    subsets = []
    actual_alloc = {}

    for raw_task, norm_task in zip(tasks, normalized_tasks):
        tds = _prepare_task_ds(dataset, norm_task, seed=seed)
        avail = len(tds)
        take = min(allocations[norm_task], avail)
        if take < allocations[norm_task]:
            print(f"Warning: Only {avail} examples available for task {raw_task}. Taking all of them.")

        if balance_value and take > 0:
            sub = _greedy_balance_pick(tds, k=take, seed=seed)
        else:
            sub = tds.select(range(take))

        actual_alloc[raw_task] = len(sub)
        subsets.append(sub.remove_columns([c for c in ["plus_cnt", "minus_cnt"] if c in sub.column_names]))

    combined = concatenate_datasets(subsets).shuffle(seed=seed)
    print("Selected counts per task:")
    for task in tasks:
        print(f"  {task}: {actual_alloc.get(task, 0)}")
    print(f"Total selected: {len(combined)} (target was {target_size})")
    return combined

def analyze_value_distribution(dataset):
    counts = Counter()
    for vals in dataset["value"]:
        # value가 list라는 가정; 다른 토큰은 무시
        counts.update(v for v in vals if v in {"+", "-"})
    total = sum(counts.values())
    print("Value token distribution:")
    for label in ["+", "-"]:
        portion = counts[label] / total if total else 0.0
        print(f"  {label}: {counts[label]} ({portion:.2%})")
    others = [k for k in counts.keys() if k not in {"+", "-"}]
    if others:
        print("Other tokens:")
        for k in others:
            portion = counts[k] / total if total else 0.0
            print(f"  {k}: {counts[k]} ({portion:.2%})")
    return counts


In [ ]:
# --------- 실행 예시 ---------
ds = load_dataset("zhuzilin/Math-Shepherd")
small_train2 = select_balanced_subset(ds["train"], balance_value=True)
value_counts2 = analyze_value_distribution(small_train2)

out_dir = "/home/leena/ccc_eval/rs_prm/data/math_shepherd_small_train.arrow"
small_train2.save_to_disk(out_dir)
reloaded_arrow = load_from_disk(out_dir)
print(reloaded_arrow)

## add gold answer

In [20]:
import re, os
from typing import Dict, Tuple, Optional, List
from datasets import Dataset
from difflib import SequenceMatcher
import sys, pathlib
ROOT = pathlib.Path().resolve().parent  # 현재 노트북이 있는 디렉토리의 상위
sys.path.append(str(ROOT))
from inference.math_scorer import MATHScorer
from inference.gsm_scorer import GSM8KScorer

# -----------------------
# Normalization & parsers
# -----------------------
def _normalize_text(s: str) -> str:
    if s is None: return ""
    s = s.replace("\u00a0", " ")
    s = re.sub(r"\s+", " ", s.strip())
    return s

def _normalize_task(name: str) -> str:
    return (name or "").strip().lower()

def _extract_question_from_ms_input(ms_input: str) -> str:
    if not isinstance(ms_input, str):
        return ""
    s = ms_input

    # 1) 첫 'Step <num>:' 이전까지
    m = re.search(r"\bStep\s*\d+\s*:", s, flags=re.IGNORECASE)
    cut = m.start() if m else None
    # 2) 보조 컷 포인트: 'The answer is:'
    m2 = re.search(r"\bThe\s+answer\s+is\s*:", s, flags=re.IGNORECASE)
    cut2 = m2.start() if m2 else None
    # 가장 이른 컷 포인트 선택
    candidates = [x for x in [cut, cut2] if x is not None]
    if candidates:
        s = s[: min(candidates)]
    # 3) 혹시 라벨이 전혀 없고 줄바꿈만 있는 경우: 첫 문단 사용
    # s = s.strip().splitlines()[0] if "\n" in s and len(s.strip().splitlines()[0]) > 20 else s
    # 4) 트레일러 토큰 정리 (마커/수식 시작 전)
    s = s.split(" ки")[0]  # 러시아어 '키' 마커가 문장 뒤에 붙는 케이스 방지
    s = s.split("<<")[0]   # 계산 마커 이전까지만
    s = _normalize_text(s)
    return s

# -----------------------
# Source indices (train)
# -----------------------
def build_gsm8k_index(gsm_train: Dataset) -> Tuple[Dict[str, str], List[str]]:
    q2a, questions = {}, []
    for ex in gsm_train:
        q = _normalize_text(ex.get("question", ""))
        a_raw = ex.get("answer", None)
        a = GSM8KScorer.extract_gold(a_raw) if isinstance(a_raw, str) else None
        if q and a:
            q2a[q] = a
            questions.append(q)
    return q2a, questions

def build_math500_index(math500_test: Dataset) -> Tuple[Dict[str, str], List[str]]:
    q2a, questions = {}, []
    for ex in math500_test:
        q = _normalize_text(ex.get("problem", ""))
        a = ex.get("answer", None)  # 이미 최종 정답
        if q and isinstance(a, str) and a.strip():
            q2a[q] = _normalize_text(a)
            questions.append(q)
    return q2a, questions

def build_math_main_index(math_train, math_test, math_scorer):
    """hendrycks/competition_math의 train+test에서 (q -> gold) 인덱스 구축"""
    def _one_split(ds):
        q2a, qs = {}, []
        for ex in ds:
            q = _normalize_text(ex.get("problem", ""))
            sol = ex.get("solution", "")
            try:
                a = math_scorer.extract_gold_answer(sol)
            except Exception:
                a = None
            if q and a is not None:
                a = _normalize_text(str(a))
                q2a[q] = a
                qs.append(q)
        return q2a, qs

    q2a_tr, qs_tr = _one_split(math_train)
    q2a_te, qs_te = _one_split(math_test)
    q2a = {**q2a_tr, **q2a_te}
    qs   = qs_tr + qs_te
    return q2a, qs

# -----------------------
# Matching
# -----------------------
def _best_fuzzy(query: str, candidates: List[str], threshold: float = 0.95) -> Optional[str]:
    best, best_r = None, 0.0
    for cand in candidates:
        r = SequenceMatcher(a=query, b=cand).ratio()
        if r > best_r:
            best, best_r = cand, r
    return best if best is not None and best_r >= threshold else None

from rapidfuzz import process, fuzz
def _best_fuzzy_rf(query: str, candidates: List[str], cutoff: int = 95) -> Optional[str]:
    res = process.extractOne(
        query,
        candidates,
        scorer=fuzz.ratio,     # difflib 유사도와 가장 근접
        score_cutoff=cutoff    # 0~100
        # processor=None  # 이미 외부에서 _normalize_text 했으니 별도 processor 불필요
    )
    return res[0] if res else None

# -----------------------------------------
# Main: attach gold_answer with fail indices
# -----------------------------------------
def attach_gold_answers_by_task(ms_ds: Dataset, gsm_index: Tuple[Dict[str, str], List[str]], math500_index: Tuple[Dict[str, str], List[str]], fuzzy_threshold: float = 0.95,) -> Tuple[Dataset, List[int]]:
    gsm_q2a, gsm_qs = gsm_index
    m5_q2a, m5_qs = math500_index
    fail_indices: List[int] = []

    def _lookup(task: str, ms_input: str) -> Tuple[Optional[str], str, str]:
        q = _extract_question_from_ms_input(ms_input)
        t = _normalize_task(task)

        if not q:
            return None, "none", "none"

        if t == "gsm8k":
            # exact
            if q in gsm_q2a:
                return gsm_q2a[q], "gsm8k", "exact"
            # fuzzy
            hit = _best_fuzzy(q, gsm_qs, threshold=fuzzy_threshold)
            if hit:
                return gsm_q2a[hit], "gsm8k", "fuzzy"
            return None, "none", "none"

        if t == "math":
            # exact
            if q in m5_q2a:
                return m5_q2a[q], "math500", "exact"
            # fuzzy
            hit = _best_fuzzy(q, m5_qs, threshold=fuzzy_threshold)
            if hit:
                return m5_q2a[hit], "math500", "fuzzy"
            return None, "none", "none"

        # 비정형 태스크면 양쪽 시도(보수적)
        if q in gsm_q2a:
            return gsm_q2a[q], "gsm8k", "exact"
        if q in m5_q2a:
            return m5_q2a[q], "math500", "exact"
        hit = _best_fuzzy(q, gsm_qs, threshold=fuzzy_threshold)
        if hit:
            return gsm_q2a[hit], "gsm8k", "fuzzy"
        hit = _best_fuzzy(q, m5_qs, threshold=fuzzy_threshold)
        if hit:
            return m5_q2a[hit], "math500", "fuzzy"
        return None, "none", "none"

    def _mapper(example, idx):
        gold, src, mtype = _lookup(example.get("task", ""), example.get("input", ""))
        if gold is None:
            fail_indices.append(idx)
        return {
            "gold_answer": gold,
            "match_source": src,
            "match_type": mtype,
        }

    enriched = ms_ds.map(_mapper, with_indices=True, desc="Adding gold_answer (GSM8K & Math500)")
    return enriched, fail_indices

def attach_gold_answers_by_task_rf(ms_ds: Dataset, gsm_index: Tuple[Dict[str, str], List[str]], math_main_index: Tuple[Dict[str, str], List[str]], math500_index: Tuple[Dict[str, str], List[str]], fuzzy_cutoff: int = 95, num_proc: int = os.cpu_count(),) -> Tuple[Dataset, List[int]]:
    gsm_q2a, gsm_qs = gsm_index
    mmain_q2a, mmain_qs   = math_main_index
    m5_q2a, m5_qs         = math500_index

    # mapper는 프로세스 간 공유상태 사용 금지 → 실패 여부는 컬럼으로 반환
    def _mapper(example, idx):
        q = _extract_question_from_ms_input(example.get("input", ""))
        t = _normalize_task(example.get("task", ""))

        gold, src, mtype = None, "none", "none"
        if q:
            if t == "gsm8k":
                if q in gsm_q2a:
                    gold, src, mtype = gsm_q2a[q], "gsm8k", "exact"
                else:
                    hit = _best_fuzzy_rf(q, gsm_qs, cutoff=fuzzy_cutoff)
                    if hit: gold, src, mtype = gsm_q2a[hit], "gsm8k", "fuzzy"

            elif t == "math":
                # 1) competition_math 우선
                if q in mmain_q2a:
                    gold, src, mtype = mmain_q2a[q], "math", "exact"
                else:
                    hit = _best_fuzzy_rf(q, mmain_qs, cutoff=fuzzy_cutoff)
                    if hit:
                        gold, src, mtype = mmain_q2a[hit], "math", "fuzzy"
                    else:
                        # 2) fallback: MATH-500
                        if q in m5_q2a:
                            gold, src, mtype = m5_q2a[q], "math500", "exact"
                        else:
                            hit = _best_fuzzy_rf(q, m5_qs, cutoff=fuzzy_cutoff)
                            if hit: gold, src, mtype = m5_q2a[hit], "math500", "fuzzy"

            else:
                # 비정형 태스크: 세 소스 모두 탐색
                if q in gsm_q2a:
                    gold, src, mtype = gsm_q2a[q], "gsm8k", "exact"
                elif q in mmain_q2a:
                    gold, src, mtype = mmain_q2a[q], "math", "exact"
                elif q in m5_q2a:
                    gold, src, mtype = m5_q2a[q], "math500", "exact"
                else:
                    for (qs, q2a, src_tag) in [(gsm_qs, gsm_q2a, "gsm8k"), (mmain_qs, mmain_q2a, "math"), (m5_qs, m5_q2a, "math500")]:
                        hit = _best_fuzzy_rf(q, qs, cutoff=fuzzy_cutoff)
                        if hit:
                            gold, src, mtype = q2a[hit], src_tag, "fuzzy"
                            break

        return {"gold_answer": gold, "match_source": src, "match_type": mtype, "match_ok": gold is not None}

    enriched = ms_ds.map(_mapper, with_indices=True, num_proc=num_proc,
                         desc=f"Adding gold_answer (RapidFuzz, cutoff={fuzzy_cutoff})")
    fail_ids = [i for i, ok in enumerate(enriched["match_ok"]) if not ok]
    enriched = enriched.remove_columns(["match_ok"])
    return enriched, fail_ids



In [ ]:
data_dir = "/home/leena/ccc_eval/rs_prm/data/math_shepherd_small_train2.arrow"
ms_small = load_from_disk(data_dir)
gsm_train = load_dataset("openai/gsm8k", "main")["train"]
math500 = load_dataset("HuggingFaceH4/MATH-500")["test"] 

gsm_index = build_gsm8k_index(gsm_train)
math500_index = build_math500_index(math500)

ms_with_gold, fail_ids = attach_gold_answers_by_task(
    ms_small, gsm_index=gsm_index, math500_index=math500_index, fuzzy_threshold=0.95
)

In [50]:
data_dir = "/home/leena/ccc_eval/rs_prm/data/ms_with_gold_all_120k"
ms_small = load_from_disk(data_dir)
ms_small[800]

{'input': 'Two complementary angles, A and B, have measures in the ratio of 7 to 23, respectively. What is the ratio of the measure of the complement of angle A to the measure of the complement of angle B? Express your answer as a common fraction. Step 1: I know that complementary angles are angles that add up to 90 degrees, so A + B = 90. ки\nStep 2: I also know that the ratio of A to B is 7 to 23, which means that A = 7k and B = 23k for some constant k. ки\nStep 3: Substituting these expressions into the equation A + B = 90, I get 7k + 23k = 90, or 30k = 90, or k = 3. ки\nStep 4: Therefore, A = 7k = 7(3) = 21 and B = 23k = 23(3) = 69. ки\nStep 5: The complement of A is 90 - A, which is 90 - 21 = 69. ки\nStep 6: The complement of B is 90 - B, which is 90 - 69 = 21. ки\nStep 7: The ratio of the complement of A to the complement of B is 69 to 21, which can be simplified by dividing both numerator and denominator by 3. The answer is: 23/7 ки',
 'label': 'Two complementary angles, A and B

In [ ]:
out_dir = "/home/leena/ccc_eval/rs_prm/data/math_shepherd_small_trainset"
ms_with_gold.save_to_disk(out_dir)

n_total = len(ms_with_gold)
n_fail = len(fail_ids)
n_exact = sum(1 for t in ms_with_gold["match_type"] if t == "exact")
n_fuzzy = sum(1 for t in ms_with_gold["match_type"] if t == "fuzzy")
print(f"[REPORT] total={n_total}, exact={n_exact}, fuzzy={n_fuzzy}, fail={n_fail}")
if n_fail:
    print(f"[REPORT] fail indices (count={n_fail}): {fail_ids[:50]}{'...' if n_fail > 50 else ''}")

In [ ]:
# calculate "+/-" mixing samples

def count_plus_minus_entries(dataset, field="value"):
    target = {"+", "-"}
    return sum(
        1
        for record in dataset
        if isinstance(record.get(field), (list, tuple, set, str))
        and target.issubset(set(record[field]))
    )

from datasets import load_from_disk
data_dir = "/home/leena/ccc_eval/rs_prm/data/ms_with_gold_all_120k"
ms_small = load_from_disk(data_dir)

# example usage
mixed_count = count_plus_minus_entries(ms_small)
print(mixed_count)

40305


# MI rewards

In [24]:
from typing import Any, Dict, Iterable, List, Optional, Tuple
import torch
import time, math, re, random
from datasets import load_dataset
from run_profile import RunProfiler  # 네가 쓰던 프로파일러

class MIHardReward:
    _STEP_RE = re.compile(r"(?:^|\s)(Step\s+\d+\s*:\s*)", flags=re.IGNORECASE)
    _ANS_RE  = re.compile(r"The\s+answer\s+is\s*:\s*(.+?)\s*(?:[+\-]\s*$|\s*$)", flags=re.IGNORECASE | re.DOTALL)

    def __init__(self, model, tokenizer):
        self.model = model
        self.tokenizer = tokenizer
        self.device = next(model.parameters()).device
        self._H_CACHE: Dict[Tuple[str, str, str], float] = {}
        self.prof = RunProfiler()

    def _clean_spaces(self, s: str) -> str:
        s = s.replace("\u200b", " ").replace("\xa0", " ")
        s = re.sub(r"[ \t]+", " ", s)
        s = re.sub(r"\s+\n", "\n", s)
        return s.strip()
    
    def _find_suffix_start(self, full_ids: List[int], tgt_ids: List[int]) -> int:
        if not tgt_ids or len(tgt_ids) > len(full_ids):
            return -1
        Lf, Lt = len(full_ids), len(tgt_ids)
        for start in range(Lf - Lt, -1, -1):
            if full_ids[start:start+Lt] == tgt_ids:
                return start
        return -1

    def build_prompt(self, question: str, tokenizer=None) -> str:
        return f"Problem:\n{question}\nSolution (step-by-step):\n"
    
    def _format_answer_target(self, gold: str) -> str:
        gold = self._clean_spaces(str(gold))
        return f"The answer is: {gold}"

    # ----------------- Math-Shepherd parsing -----------------
    def parse_math_shepherd_record(self, rec: Dict[str, Any]) -> Dict[str, Any]:
        txt = rec.get("label")
        txt = self._clean_spaces(txt)

        # 문제/풀이 분리
        m_first = re.search(r"Step\s+1\s*:", txt, flags=re.IGNORECASE)
        if m_first:
            question = txt[:m_first.start()].strip()
            tail = txt[m_first.start():].strip()
        else:
            question = re.sub(r"The\s+answer\s+is\s*:.*$", "", txt, flags=re.IGNORECASE).strip()
            tail = ""

        # 정답 파싱
        gold_answer = ""
        answer_target = ""
        ma = self._ANS_RE.search(txt)
        if ma:
            body = self._clean_spaces(ma.group(1))
            body = re.sub(r"\s*[+\-]\s*$", "", body)  # 끝의 +/- 정리
            gold_answer = body
            answer_target = self._format_answer_target(body)

        # ---------- NEW: dataset에 붙인 gold_answer가 있으면 무조건 우선 ----------
        ds_gold = rec.get("gold_answer", None)
        if isinstance(ds_gold, str):
            ds_gold = self._clean_spaces(ds_gold)
        if ds_gold:  # gold가 있으면 라벨 기반 추출을 덮어씀
            gold_answer = ds_gold
            answer_target = self._format_answer_target(ds_gold)

        steps: List[str] = []
        step_labels_pm: List[str] = []

        parts = self._STEP_RE.split(tail)
        for idx in range(1, len(parts), 2):
            step_tag = parts[idx]
            after = parts[idx+1] if idx+1 < len(parts) else ""

            # 이번 스텝 본문 경계
            mnext = self._STEP_RE.search(after)
            mans  = re.search(r"The\s+answer\s+is\s*:", after, flags=re.IGNORECASE)
            if mnext and mans:
                next_cut = min(mnext.start(), mans.start())
            elif mnext:
                next_cut = mnext.start()
            elif mans:
                next_cut = mans.start()
            else:
                next_cut = len(after)

            # 본문 후보
            step_text = (step_tag + " " + after[:next_cut]).strip()

            # 1) 스텝 본문 끝에서 우선 +/- 탐지
            pm_in_body = re.search(r"([+\-])\s*$", step_text)
            if pm_in_body:
                pm = pm_in_body.group(1)
                step_text = re.sub(r"[+\-]\s*$", "", step_text).rstrip()
            else:
                # 2) 본문 뒤 suffix의 맨 앞에서 +/- 탐지 (기존 로직)
                suffix = after[next_cut:].lstrip()
                if   suffix.startswith("+"): pm = "+"
                elif suffix.startswith("-"): pm = "-"
                else:                        pm = "+"

            steps.append(step_text)     # <-- 이미 +/− 제거된 클린 스텝
            step_labels_pm.append(pm)

        step_values = rec.get("value")
        if len(step_values) != len(steps):
            L = min(len(step_values), len(steps))
            steps = steps[:L]
            step_values = step_values[:L]
        correct_mask = [1 if pm == "+" else 0 for pm in step_values]
        
        return {
            "question": question,
            "steps": steps,                 # +/− 제거된 본문만
            "step_pm": step_labels_pm,      # 원본 라벨
            "correct_mask": correct_mask,   # +→1, −→0
            "gold_answer": gold_answer,     # ★ 여기 최종 gold (우선순위: rec.gold → 라벨 추출)
            "answer_target": answer_target, # "The answer is: <gold>"
            "task": rec.get("task", None),
            "raw": txt,
        }

    # ----------------- entropy / MI -----------------
    @torch.no_grad()
    def _entropy_bits_total(self, prompt: str, target: str) -> Tuple[float, float, int]:
        """Return (H_total_bits, H_bits_per_token, target_len) for H(A | prompt)."""
        t0 = time.perf_counter()
        full = prompt + target
        full_enc = self.tokenizer(full, return_tensors="pt", add_special_tokens=True).to(self.device)
        tgt_ids  = self.tokenizer(target, add_special_tokens=False)["input_ids"]
        input_ids = full_enc["input_ids"]
        full_ids = input_ids[0].tolist()
        L_full, L_tgt = len(full_ids), len(tgt_ids)

        if L_tgt == 0:
            wall = time.perf_counter() - t0
            try:
                self.prof.log(tag="mi:entropy:call", backend="hf", wall_s=wall, gen_tokens=0, prompt_len=L_full if 'L_full' in locals() else 0, target_len=0)
            except Exception:
                pass
            return 0.0, 0.0, 0

        Lp = self._find_suffix_start(full_ids, tgt_ids)
        if Lp < 0:
            Lp = L_full - L_tgt

        logits = self.model(**full_enc).logits.float()      # [1, L, V]
        logits_shifted = logits[:, :-1, :]                  # [1, L-1, V]

        start = max(Lp - 1, 0)
        end   = min(start + L_tgt, logits_shifted.shape[1])
        eff_len = end - start
        if eff_len <= 0:
            wall = time.perf_counter() - t0
            try:
                self.prof.log(tag="mi:entropy:call", backend="hf", wall_s=wall, gen_tokens=0, prompt_len=L_full, target_len=L_tgt)
            except Exception:
                pass
            return 0.0, 0.0, 0

        LOG2E = 1.0 / math.log(2.0)
        lp = torch.log_softmax(logits_shifted[0, start:end, :], dim=-1)
        H_bits_per_step = (-(lp.exp() * lp).sum(dim=-1)) * LOG2E
        H_bits_sum = float(H_bits_per_step.sum().item())
        H_bits_mean = H_bits_sum / eff_len

        wall = time.perf_counter() - t0
        try:
            self.prof.log(tag="mi:entropy:call", backend="hf", wall_s=wall, gen_tokens=eff_len, prompt_len=L_full, target_len=L_tgt)
        except Exception:
            pass
        return H_bits_sum, H_bits_mean, eff_len

    def _H_cached(self, prompt: str, target: str) -> float:
        if not hasattr(self, "_H_CACHE"):
            self._H_CACHE = {}
        key = ("H|", prompt, "\u241E", target)
        if key in self._H_CACHE:
            return self._H_CACHE[key]
        H, _, _ = self._entropy_bits_total(prompt, target)
        self._H_CACHE[key] = H
        return H
    
    # ----------------- MI algorithms -----------------
    def compute_step_mi_loo(self, question: str, steps: List[str], answer_target: str, tokenizer):
        """Leave-one-out contribution: Δ_i = H(all without i) - H(all). Returns a List[float] with each step's Δ_i."""
        t0 = time.perf_counter()
        question = re.sub(r' +', ' ', question)
        base = self.build_prompt(question, tokenizer=tokenizer) + "\n"
        with_all = base + "".join(s.rstrip().rstrip("\n") + "\n" for s in steps)
        H_all = self._H_cached(with_all, answer_target)

        print("\n[LOO] ----------")
        print("[LOO] Base prompt:\n", base, sep="")
        print("[LOO] Answer target:\n", answer_target, sep="")
        print("[LOO] Steps:")
        for k, s in enumerate(steps):
            print(f"  [{k}] {s[:200]}{'...' if len(s)>200 else ''}")
        print("[LOO] H_all:", H_all)

        contribs = []
        for i in range(len(steps)):
            without_i = base + "".join(steps[j].rstrip().rstrip("\n") + "\n" for j in range(len(steps)) if j != i)
            H_wo = self._H_cached(without_i, answer_target)
            contribs.append(H_wo - H_all)
            print(f"[LOO] without_i", without_i)
            print(f"[LOO] drop step {i} -> H_wo={H_wo:.4f}, contrib={H_wo - H_all:.4f}")
        
        try: # profile
            self.prof.log(tag="mi:loo:call", backend="meta", num_prompts=len(steps) + 1, n=1, wall_s=time.perf_counter() - t0, num_steps=len(steps))
        except Exception:
            pass
        return contribs

    def compute_step_mi_marginal(self, question: str, steps: List[str], answer_target: str, tokenizer):
        """marginal effect of step: MI_i = H(base) - H(base + S_i) order dependent ↓: List[float]"""
        t0 = time.perf_counter()
        question = re.sub(r' +', ' ', question)
        base = self.build_prompt(question, tokenizer=tokenizer) + "\n"
        H_base = self._H_cached(base, answer_target)

        print("\n[MARGINAL] ----------")
        print("[MARGINAL] Base prompt:\n", base, sep="")
        print("[MARGINAL] Answer target:\n", answer_target, sep="")
        print(f"[MARGINAL] H_base={H_base:.4f}")

        mis = []
        for s in steps:
            with_i = base + s.rstrip().rstrip("\n") + "\n"
            H_with = self._H_cached(with_i, answer_target)
            mis.append(H_base - H_with)
            print(f"[MARGINAL] with_i", with_i)
            print(f"[MARGINAL] add step -> H_with={H_with:.4f}, mi={H_base - H_with:.4f}")
        try:
            self.prof.log(tag="mi:marginal:call", backend="meta", num_prompts=len(steps) + 1, n=1, wall_s=time.perf_counter() - t0, num_steps=len(steps))
        except Exception:
            pass
        return mis

    def compute_step_mi_shapley(self, question: str, steps: List[str], answer_target: str, tokenizer, n_perm: int = 16, seed: int = 42):
        """Shapley approximation: various random permutation π's average Δ_MI  φ_i ≈ E_π[ H(base + prefix_before_i) - H(base + prefix_before_i + S_i) ]"""
        t0 = time.perf_counter()
        question = re.sub(r' +', ' ', question)
        base = self.build_prompt(question, tokenizer=tokenizer) + "\n"

        print("\n[SHAPLEY] ----------")
        print("[SHAPLEY] Base prompt:\n", base, sep="")
        print("[SHAPLEY] Answer target:\n", answer_target, sep="")
        print(f"[SHAPLEY] n_perm={n_perm}, seed={seed}")

        N = len(steps)
        rng = random.Random(seed)
        shap = [0.0] * N
        for _ in range(n_perm):
            idxs = list(range(N))
            rng.shuffle(idxs)
            prompt = base
            H_prev = self._H_cached(prompt, answer_target)
            for idx in idxs:
                prompt_with = prompt + steps[idx].rstrip().rstrip("\n") + "\n"
                H_with = self._H_cached(prompt_with, answer_target)
                shap[idx] += (H_prev - H_with)
                prompt, H_prev = prompt_with, H_with
        shap = [v / n_perm for v in shap]
        print("[SHAPLEY] shapley:", [round(v, 4) for v in shap])
        try:
            self.prof.log(tag="mi:shapley:call", backend="meta", num_prompts=n_perm * (N + 1), n=1, wall_s=time.perf_counter() - t0, num_steps=N, n_perm=n_perm)
        except Exception:
            pass
        return shap

    def compute_step_mi_cmi(self, question: str, steps: List[str], answer_target: str, tokenizer) -> List[float]:
        """Sequential conditional MI (no permutations): For each step i in the given order, compute CMI_i = H(base + prefix_<i>) - H(base + prefix_≤i) where H(·) is the total bits of H(A | prompt), and prefix_<i> is the concatenation of steps up to but not including i."""
        t0 = time.perf_counter()
        question = re.sub(r' +', ' ', question)
        base = self.build_prompt(question, tokenizer=tokenizer) + "\n"

        print("\n[CMI] ----------")
        print("[CMI] Base prompt:\n", base, sep="")
        print("[CMI] Answer target:\n", answer_target, sep="")

        vals: List[float] = []
        prompt = base
        H_prev = self._H_cached(prompt, answer_target)
        for s in steps:
            prompt_with = prompt + s.rstrip().rstrip("\n") + "\n"
            H_with = self._H_cached(prompt_with, answer_target)
            vals.append(H_prev - H_with)
            print(f"[CMI] step: H_prev={H_prev:.4f} -> H_with={H_with:.4f}, cmi={H_prev - H_with:.4f}")
            prompt = prompt_with
            H_prev = H_with
        try:
            self.prof.log(tag="mi:cmi:call", backend="meta", num_prompts=len(steps) + 1, n=1, wall_s=time.perf_counter() - t0, num_steps=len(steps))
        except Exception:
            pass
        return vals
    
    # ----------------- public: labeling (stream) -----------------
    def mi_labelling(self, *, ds, n_shapley_perm: int = 16, seed: int = 42, ds_task_tag: Optional[str] = None) -> Iterable[Dict[str, Any]]:
        t_ds0 = time.perf_counter()
        for si, rec in enumerate(ds):
            t0 = time.perf_counter()
            parsed = self.parse_math_shepherd_record(rec)
            question      = parsed["question"]
            steps         = parsed["steps"]
            gold_answer   = parsed["gold_answer"]
            answer_target = self._format_answer_target(gold_answer) if gold_answer else ""
            correct_mask  = parsed["correct_mask"]
            task          = parsed.get("task") or ds_task_tag or "mathshepherd"

            print("\n[Example] Parsed keys:", list(parsed.keys()))
            print("[Example] question:", (parsed["question"] or ""))
            print("[Example] answer_target:", parsed["answer_target"])

            if not question or not steps or not answer_target:
                try:
                    self.prof.log(tag="skip_empty", backend="meta", dataset=task, sample_idx=si)
                except Exception:
                    pass
                continue

            # MI Calculation
            # LOO
            try:
                if torch.cuda.is_available():
                    try: torch.cuda.reset_peak_memory_stats()
                    except Exception: pass
                t1 = time.perf_counter()
                mi_loo = self.compute_step_mi_loo(question, steps, answer_target, tokenizer=self.tokenizer)
                wall = time.perf_counter() - t1
                peak = None
                if torch.cuda.is_available():
                    try: peak = torch.cuda.max_memory_allocated() / (1024**3)
                    except Exception: pass
                self.prof.log(tag="mi:loo:sample", backend="hf", wall_s=wall, peak_mem_gb=peak, dataset=task, sample_idx=si, num_steps=len(steps))
            except Exception:
                mi_loo = self.compute_step_mi_loo(question, steps, answer_target, tokenizer=self.tokenizer)

            # Shapley
            try:
                if torch.cuda.is_available():
                    try: torch.cuda.reset_peak_memory_stats()
                    except Exception: pass
                t1 = time.perf_counter()
                mi_shapley = self.compute_step_mi_shapley(question, steps, answer_target, tokenizer=self.tokenizer, n_perm=n_shapley_perm, seed=seed)
                wall = time.perf_counter() - t1
                peak = None
                if torch.cuda.is_available():
                    try: peak = torch.cuda.max_memory_allocated() / (1024**3)
                    except Exception: pass
                self.prof.log(tag="mi:shapley:sample", backend="hf", wall_s=wall, peak_mem_gb=peak, dataset=task, sample_idx=si, num_steps=len(steps))
            except Exception:
                mi_shapley = self.compute_step_mi_shapley(question, steps, answer_target, tokenizer=self.tokenizer, n_perm=n_shapley_perm, seed=seed)

            # CMI
            try:
                if torch.cuda.is_available():
                    try: torch.cuda.reset_peak_memory_stats()
                    except Exception: pass
                t1 = time.perf_counter()
                mi_cmi = self.compute_step_mi_cmi(question, steps, answer_target, tokenizer=self.tokenizer)
                wall = time.perf_counter() - t1
                peak = None
                if torch.cuda.is_available():
                    try: peak = torch.cuda.max_memory_allocated() / (1024**3)
                    except Exception: pass
                self.prof.log(tag="mi:cmi:sample", backend="hf", wall_s=wall, peak_mem_gb=peak, dataset=task, sample_idx=si, num_steps=len(steps))
            except Exception:
                mi_cmi = self.compute_step_mi_cmi(question, steps, answer_target, tokenizer=self.tokenizer)

            # Marginal
            try:
                if torch.cuda.is_available():
                    try: torch.cuda.reset_peak_memory_stats()
                    except Exception: pass
                t1 = time.perf_counter()
                mi_margin = self.compute_step_mi_marginal(question, steps, answer_target, tokenizer=self.tokenizer)
                wall = time.perf_counter() - t1
                peak = None
                if torch.cuda.is_available():
                    try: peak = torch.cuda.max_memory_allocated() / (1024**3)
                    except Exception: pass
                self.prof.log(tag="mi:margin:sample", backend="hf", wall_s=wall, peak_mem_gb=peak, dataset=task, sample_idx=si, num_steps=len(steps))
            except Exception:
                mi_margin = self.compute_step_mi_marginal(question, steps, answer_target, tokenizer=self.tokenizer)

            entry = {
                "question": question,
                "completion": steps,            # 원본 step들
                "original_answer": gold_answer,     # 본문만(참고용)
                "answer_target": answer_target, # "The answer is: …" (MI 타깃)
                "mi_loo": mi_loo,
                "mi_shapley": mi_shapley,
                "mi_cmi": mi_cmi,
                "mi_margin": mi_margin,
                "correct_mask": correct_mask,
                "task": task,
            }
            yield entry

            try:
                self.prof.log(tag="sample_total", backend="meta", dataset=task, sample_idx=si, wall_s=time.perf_counter() - t0)
            except Exception:
                pass

        # final profile log
        try:
            self.prof.log(tag="dataset_total", backend="meta", wall_s=time.perf_counter() - t_ds0)
        except Exception:
            pass
    

In [47]:
# --- 아주 작은 모델로 스모크 테스트 ---
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_from_disk

model_name = "Qwen/Qwen3-4B-base"
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)
mi = MIHardReward(model=model, tokenizer=tokenizer)
data_dir = "/home/leena/ccc_eval/rs_prm/data/ms_with_gold_all_120k"
reloaded_arrow = load_from_disk(data_dir)
toy_ds = reloaded_arrow.select(range(10,11))

print("== Tiny model loaded ==")
for i, entry in enumerate(mi.mi_labelling(ds=toy_ds, n_shapley_perm=8)):
    print("\n=== ENTRY", i, "===")
    print("Q:", entry["question"])
    print("Answer target:", entry["answer_target"])
    print("Steps:", len(entry["completion"]))
    print("correct_mask:", entry["correct_mask"])
    print("mi_loo:", [round(x, 4) for x in entry["mi_loo"]])
    print("mi_margin:", [round(x, 4) for x in entry["mi_margin"]])
    print("mi_cmi:", [round(x, 4) for x in entry["mi_cmi"]])
    print("mi_shapley:", [round(x, 4) for x in entry["mi_shapley"]])

In [ ]:
import json, os

def read_jsonl(file_path):
    data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                data.append(json.loads(line))
    return data

def jsonl_to_json(jsonl_path, json_path):
    data = read_jsonl(jsonl_path)
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    print(f"Converted {jsonl_path} to {json_path}")

def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model_name = "Qwen/Qwen3-4B-base"  # "Qwen/Qwen2.5-Math-7B-Instruct"  
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        device_map="auto",     
        torch_dtype="auto",   
        trust_remote_code=True 
    )
    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=False)
    mi = MIHardReward(model=model, tokenizer=tokenizer)
    print("Finish loading model and tokenizer")

    data_dir = "/home/leena/ccc_eval/rs_prm/data/math_shepherd_small_train2.arrow"
    small_train2.save_to_disk(data_dir)
    ds = load_from_disk(data_dir)
    print(f"Dataset loaded: {len(ds)} rows")

    out_dir = "/home/leena/ccc_eval/rs_prm/samples/mi2"
    os.makedirs(out_dir, exist_ok=True)
    output_file = os.path.join(out_dir, f"ms_small_mi_qw3_4b.jsonl")
    profile_txt = output_file + "_profile.txt"
    profile_json = output_file + "_profile.json"

    with open(output_file, "w", encoding="utf-8") as f:
        for i, entry in enumerate(mi.mi_labelling(ds=ds)):
            f.write(json.dumps(entry, ensure_ascii=False) + "\n")
            f.flush() 

    print(f"Data saved to {output_file}")
    json_path = output_file[:-6] + ".json"
    jsonl_to_json(output_file, json_path)

    header = f"MathShepherd/mi :: model={model_name}"
    mi.prof.dump_summary_text(profile_txt, header=header)
    mi.prof.dump_summary_json(profile_json, header=header)
    print(f"Profile (text)  -> {profile_txt}")
    print(f"Profile (json)  -> {profile_json}")

# Merge Steps

## Normalization

In [9]:
import json, argparse, sys, math
from pathlib import Path
from typing import List, Dict, Any, Tuple, Optional
from difflib import SequenceMatcher
from collections import defaultdict
import numpy as np

# -----------------------------
# IO helpers
# -----------------------------
def _load_records(path: str) -> List[Dict[str, Any]]:
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(path)
    text = p.read_text(encoding="utf-8").strip()
    # Heuristics: if startswith '[' -> JSON array
    if text.startswith("["):
        data = json.loads(text)
        if not isinstance(data, list):
            raise ValueError("JSON file must contain an array.")
        return data
    # else assume JSONL
    recs = []
    for ln in text.splitlines():
        ln = ln.strip()
        if not ln:
            continue
        recs.append(json.loads(ln))
    return recs

def _dump_records(path: str, recs: List[Dict[str, Any]], json_array: bool=True) -> None:
    p = Path(path)
    p.parent.mkdir(parents=True, exist_ok=True)
    if json_array:
        p.write_text(json.dumps(recs, ensure_ascii=False, indent=2), encoding="utf-8")
    else:
        with p.open("w", encoding="utf-8") as f:
            for r in recs:
                f.write(json.dumps(r, ensure_ascii=False) + "\n")

# -----------------------------
# Text normalization & keys
# -----------------------------
def _norm_txt(s: str) -> str:
    # 보수적 정규화: 공백 표준화, 양끝 공백 제거, \n->space, 연속 스페이스 1개로
    if not isinstance(s, str):
        s = "" if s is None else str(s)
    s = s.replace("\r\n", "\n").replace("\r", "\n")
    s = " ".join(s.split())  # collapse whitespace incl. newlines/tabs
    return s

def _norm_steps(steps: List[str]) -> List[str]:
    if not isinstance(steps, list):
        return []
    return [_norm_txt(x) for x in steps]

def _make_key(entry: Dict[str, Any]) -> Tuple[str, Tuple[str, ...]]:
    q = _norm_txt(entry.get("question", ""))
    comp = entry.get("completion", [])
    comp_norm = _norm_steps(comp)
    return (q, tuple(comp_norm))

# -----------------------------
# Index builders
# -----------------------------
def _build_exact_index(recs: List[Dict[str, Any]]) -> Dict[Tuple[str, Tuple[str, ...]], Dict[str, Any]]:
    idx = {}
    for r in recs:
        idx[_make_key(r)] = r
    return idx

def _build_question_buckets(recs: List[Dict[str, Any]]) -> Dict[str, List[Dict[str, Any]]]:
    buckets = defaultdict(list)
    for r in recs:
        q = _norm_txt(r.get("question", ""))
        buckets[q].append(r)
    return buckets

# -----------------------------
# MI normalization (robust z -> [0,1])
# -----------------------------
def _percentile(xs, p):
    if not xs:
        return 0.0
    xs = sorted(xs)
    k = (len(xs)-1) * (p/100.0)
    f = math.floor(k)
    c = math.ceil(k)
    if f == c:
        return xs[int(k)]
    return xs[f] + (xs[c]-xs[f]) * (k - f)

def _median(xs):
    return _percentile(xs, 50)

def _mad(xs, med):
    # Median absolute deviation
    return _median([abs(x - med) for x in xs])

def _robust_scale(stats_vals):
    """Compute robust scale using MAD; fall back to IQR or std if necessary. Returns (median, scale, used)"""
    if not stats_vals:
        return 0.0, 1.0, "fallback:empty"

    med = _median(stats_vals)
    mad = _mad(stats_vals, med)
    # 1.4826: make MAD consistent with std for normal dist
    scale = 1.4826 * mad
    used = "mad"

    if scale <= 1e-12:
        # fallback to IQR
        q75 = _percentile(stats_vals, 75)
        q25 = _percentile(stats_vals, 25)
        iqr = max(q75 - q25, 0.0)
        # 0.7413 is approx factor s.t. 0.7413*IQR ≈ std for normal
        scale = 0.7413 * iqr
        used = "iqr"
        if scale <= 1e-12:
            # final fallback to std
            mu = sum(stats_vals)/len(stats_vals)
            var = sum((x-mu)**2 for x in stats_vals)/max(len(stats_vals)-1, 1)
            scale = math.sqrt(max(var, 1e-24))
            used = "std"
            if scale <= 1e-12:
                scale = 1.0
                used = "epsilon"
    return med, scale, used

def _map_to_unit(z, method="sigmoid"):
    if method == "sigmoid":
        # logistic; centered at 0 -> 0.5
        return 1.0 / (1.0 + math.exp(-z))
    elif method in ("cdf", "gaussian_cdf", "norm_cdf"):
        # standard normal CDF via erf
        return 0.5 * (1.0 + math.erf(z / math.sqrt(2.0)))
    else:
        raise ValueError(f"unknown map method: {method}")

def collect_mi_tail_values(recs: List[Dict[str, Any]], switch_at: int) -> List[float]:
    vals = []
    for r in recs:
        rewards = r.get("step_reward", [])
        if not isinstance(rewards, list):
            continue
        for i in range(switch_at, len(rewards)):
            v = rewards[i]
            if isinstance(v, (int, float)) and math.isfinite(v):
                vals.append(float(v))
    return vals

def normalize_mi_tail_rewards_in_memory(
    recs: List[Dict[str, Any]],
    switch_at: int = 1,
    z_clip: float = 6.0,
    map_method: str = "sigmoid",
    write_meta: bool = True
):
    mi_vals = collect_mi_tail_values(recs, switch_at)
    med, scale, scale_src = _robust_scale(mi_vals)

    out = []
    for r in recs:
        rewards = r.get("step_reward", [])
        if not isinstance(rewards, list) or len(rewards) == 0:
            out.append(r)
            continue
        new_rewards = list(rewards)
        for i in range(min(len(rewards), switch_at), len(rewards)):
            x = rewards[i]
            if isinstance(x, (int, float)) and math.isfinite(x):
                z = (float(x) - med) / (scale if scale > 0 else 1.0)
                if z_clip is not None and z_clip > 0:
                    z = max(min(z, z_clip), -z_clip)
                new_rewards[i] = _map_to_unit(z, map_method)
        new_rec = dict(r)
        new_rec["step_reward"] = new_rewards
        if write_meta:
            meta = dict(new_rec.get("meta", {}))
            meta.update({
                "mi_tail_normalization": {
                    "type": "robust_z",
                    "median": med,
                    "scale": scale,
                    "scale_source": scale_src,
                    "z_clip": z_clip,
                    "map": map_method,
                    "switch_at": switch_at,
                    "n_tail_values": len(mi_vals)
                }
            })
            new_rec["meta"] = meta
        out.append(new_rec)

    info = {
        "median": med,
        "scale": scale,
        "scale_source": scale_src,
        "z_clip": z_clip,
        "map": map_method,
        "switch_at": switch_at,
        "n_tail_values": len(mi_vals)
    }
    return out, info

# -----------------------------
# Reward selection & merging
# -----------------------------
def _normalize_reward_type(rt: str) -> str:
    rt = (rt or "").strip()
    lower = rt.lower()
    # 이미 접두사가 있으면 그대로 사용
    if lower.startswith(("mi_", "pll_", "cpmi_")):
        return rt
    # 접두사 없는 alias만 mi_로 보정 (loo, shapley, cmi, margin 등)
    if lower in {"loo", "shapley", "cmi", "margin"}:
        return f"mi_{lower}"
    # 그 외에는 건드리지 않음
    return rt

def _get_list(entry: Dict[str, Any], key: str) -> List[float]:
    val = entry.get(key, [])
    return val if isinstance(val, list) else []

def _pick_answer(entry: Dict[str, Any]) -> Optional[str]:
    # 선호도: gold_answer > answer > None
    for k in ("gold_answer", "answer"):
        if k in entry and isinstance(entry[k], str):
            return entry[k]
    return None

def _sequence_similarity(a_steps: List[str], b_steps: List[str]) -> float:
    # 간단 유사도: 두 completion 전체 문자열을 이어 붙여 비교
    a = "\n".join(a_steps)
    b = "\n".join(b_steps)
    return SequenceMatcher(a=a, b=b).ratio()

def _merge_one(
    base_rec: Dict[str, Any],
    mi_rec: Dict[str, Any],
    reward_type: str,
    on_length: str = "trim",  # 'trim' or 'strict'
    switch_at: int = 1 
) -> Dict[str, Any]:
    steps = base_rec.get("completion", [])
    steps = steps if isinstance(steps, list) else []
    n_base_steps = len(steps)

    base_rewards = _get_list(base_rec, "base_reward")
    mi_rewards   = _get_list(mi_rec, reward_type)

    # 길이 처리
    if on_length == "strict":
        if not (len(base_rewards) == len(mi_rewards) == n_base_steps):
            raise ValueError("Length mismatch in strict mode.")
        n = n_base_steps
    else:
        # trim to min length among [steps, base_rewards, mi_rewards]
        n = min(n_base_steps, len(base_rewards), len(mi_rewards) if len(mi_rewards)>0 else n_base_steps)

    if n == 0:
        merged = []
    else:
        take_base = min(switch_at, n)
        merged = base_rewards[:take_base] 
        if n > take_base:
            merged.extend(mi_rewards[take_base:n])

    # correct_mask는 가능한 쪽을 사용 (mi가 좀 더 최신 라벨일 수 있다고 가정)
    correct_mask = None
    if isinstance(mi_rec.get("correct_mask"), list):
        correct_mask = mi_rec["correct_mask"][:n]
    elif isinstance(base_rec.get("correct_mask"), list):
        correct_mask = base_rec["correct_mask"][:n]

    # answer 선택
    ans = _pick_answer(base_rec) or _pick_answer(mi_rec)

    out = {
        "question": base_rec.get("question"),
        "completion": steps[:n],
        "step_reward": merged,
        "reward_type": f"{switch_at}_qval_then_{reward_type}",
        "task": base_rec.get("task") or mi_rec.get("task"),
        "answer": ans,
        "meta": {
            "base_reward_key": "base_reward",
            "mi_reward_key": reward_type,
            "policy": f"{switch_at}_qval_then_{reward_type}",
            "trimmed_to": n,
            "base_steps_len": n_base_steps,
            "base_reward_len": len(base_rewards),
            "mi_reward_len": len(mi_rewards),
            "source_match": "exact"
        }
    }
    if correct_mask is not None:
        out["correct_mask"] = correct_mask
    return out

# -----------------------------
# Main merge routine
# -----------------------------
def merge_datasets(
    base_path: str,
    mi_path: str,
    reward_type: str,
    on_length: str="strict",
    fuzzy_threshold: float=0.95,
    switch_at: int = 1, 
) -> List[Dict[str, Any]]:
    reward_type = _normalize_reward_type(reward_type)
    base = _load_records(base_path)
    mi   = _load_records(mi_path)

    mi_exact_idx = _build_exact_index(mi)
    mi_q_buckets = _build_question_buckets(mi)

    merged: List[Dict[str, Any]] = []
    n_exact, n_fuzzy, n_missing, n_len_warn = 0, 0, 0, 0

    for b in base:
        key = _make_key(b)
        m = mi_exact_idx.get(key)
        match_mode = "exact"

        if m is None:
            q = _norm_txt(b.get("question", ""))
            candidates = mi_q_buckets.get(q, [])
            if candidates:
                b_steps = _norm_steps(b.get("completion", []))
                best, best_score = None, -1.0
                for cand in candidates:
                    score = _sequence_similarity(b_steps, _norm_steps(cand.get("completion", [])))
                    if score > best_score:
                        best, best_score = cand, score
                if best and best_score >= fuzzy_threshold:
                    m = best
                    match_mode = f"fuzzy({best_score:.3f})"

        if m is None:
            n_missing += 1
            # 매치 실패: base만으로라도 첫 스텝만 살리고 나머지는 비움(혹은 스킵)
            base_rewards = _get_list(b, "base_reward")
            steps = b.get("completion", [])
            steps = steps if isinstance(steps, list) else []
            if steps and base_rewards:
                take = min(len(steps), len(base_rewards), switch_at) 
                merged_rec = {
                    "question": b.get("question"),
                    "completion": steps[:take],
                    "step_reward": base_rewards[:take],
                    "reward_type": f"only_base_first_{take}_step_matched",
                    "task": b.get("task"),
                    "answer": _pick_answer(b),
                    "meta": {
                        "policy": "no_mi_found_keep_first_base_step",
                        "trimmed_to": take,
                        "base_steps_len": len(steps),
                        "base_reward_len": len(base_rewards),
                        "mi_reward_len": 0,
                        "source_match": "none"
                    }
                }
                if isinstance(b.get("correct_mask"), list):
                    merged_rec["correct_mask"] = b["correct_mask"][:1]
                merged.append(merged_rec)
            else:
                continue
        else:
            out = _merge_one(b, m, reward_type=reward_type, on_length=on_length, switch_at=switch_at)
            if out["meta"]["trimmed_to"] != len(b.get("completion", []) ):
                n_len_warn += 1
            out["meta"]["source_match"] = match_mode
            merged.append(out)
            if match_mode.startswith("exact"):
                n_exact += 1
            else:
                n_fuzzy += 1

    # 간단 리포트 출력
    sys.stderr.write(
        f"[merge] total_base={len(base)} matched_exact={n_exact} "
        f"matched_fuzzy={n_fuzzy} missing={n_missing} length_trimmed={n_len_warn}\n"
    )
    return merged

# -----------------------------
# CLI
# -----------------------------
def main():
    reward_type = "cpmi_marg"  # cpmi_marg | cpmi_loo | cpmi_cmi | pll_cmi_cont | pll_loo_cont | pll_marginal_cont
    fuzzy_threshold = 0.95
    on_length = "strict"    # strict | trim
    switch_at = 1

    base_path = "/home/leena/rs_prm/datasets/hard_80k/qw8b/pav/ms_pav_qw8b.json"
    mi_path = "/home/leena/rs_prm/datasets/hard_80k/qw8b/cpmi_sim/ms_cpmi_sim_qw8b.json"
    
    # 1) merge
    out_path = f"/home/leena/rs_prm/datasets/hard_80k/qw8b/pav_cpmi/pav_{reward_type}_{switch_at}_qw8b_raw.json"
    recs = merge_datasets(
        base_path=base_path,
        mi_path=mi_path,
        reward_type=reward_type,
        on_length=on_length,
        fuzzy_threshold=fuzzy_threshold,
        switch_at=switch_at,
    )
    _dump_records(out_path, recs)

    # 2) normalize MI-tail on the merged dataset (in-memory) and overwrite/alternate path
    norm_out_path = out_path.replace(".json", "_norm.json")
    norm_recs, info = normalize_mi_tail_rewards_in_memory(
        recs,
        switch_at=switch_at,
        z_clip=6.0,
        map_method="sigmoid",
        write_meta=True
    )
    _dump_records(norm_out_path, norm_recs)  # 기존 dump 그대로 사용
    print("[norm] stats:", json.dumps(info, ensure_ascii=False))


In [10]:
if __name__ == "__main__":
    main()

[merge] total_base=79999 matched_exact=79999 matched_fuzzy=0 missing=0 length_trimmed=0


[norm] stats: {"median": -0.14087327140750305, "scale": 0.2917881786684828, "scale_source": "mad", "z_clip": 6.0, "map": "sigmoid", "switch_at": 1, "n_tail_values": 401467}


In [2]:
def collect_mi_tail_values(recs: List[Dict[str, Any]], switch_at: int) -> List[float]:
    vals = []
    for r in recs:
        rewards = r.get("base_reward", [])
        if not isinstance(rewards, list):
            continue
        for i in range(switch_at, len(rewards)):
            v = rewards[i]
            if isinstance(v, (int, float)) and math.isfinite(v):
                vals.append(float(v))
    return vals

def normalize_mi_tail_rewards_in_memory(
    recs: List[Dict[str, Any]],
    switch_at: int = 1,
    z_clip: float = 6.0,
    map_method: str = "sigmoid",
    write_meta: bool = True
):
    mi_vals = collect_mi_tail_values(recs, switch_at)
    med, scale, scale_src = _robust_scale(mi_vals)

    out = []
    for r in recs:
        rewards = r.get("base_reward", [])
        if not isinstance(rewards, list) or len(rewards) == 0:
            out.append(r)
            continue
        new_rewards = list(rewards)
        for i in range(min(len(rewards), switch_at), len(rewards)):
            x = rewards[i]
            if isinstance(x, (int, float)) and math.isfinite(x):
                z = (float(x) - med) / (scale if scale > 0 else 1.0)
                if z_clip is not None and z_clip > 0:
                    z = max(min(z, z_clip), -z_clip)
                new_rewards[i] = _map_to_unit(z, map_method)
        new_rec = dict(r)
        new_rec["step_reward"] = new_rewards
        if write_meta:
            meta = dict(new_rec.get("meta", {}))
            meta.update({
                "mi_tail_normalization": {
                    "type": "robust_z",
                    "median": med,
                    "scale": scale,
                    "scale_source": scale_src,
                    "z_clip": z_clip,
                    "map": map_method,
                    "switch_at": switch_at,
                    "n_tail_values": len(mi_vals)
                }
            })
            new_rec["meta"] = meta
        out.append(new_rec)

    info = {
        "median": med,
        "scale": scale,
        "scale_source": scale_src,
        "z_clip": z_clip,
        "map": map_method,
        "switch_at": switch_at,
        "n_tail_values": len(mi_vals)
    }
    return out, info

out_path = f"/home/leena/rs_prm/datasets/hard_80k/pav/ms_pav_qw3_4b.json"
recs = _load_records(out_path)
norm_out_path = out_path.replace(".json", "_norm.json")
norm_recs, info = normalize_mi_tail_rewards_in_memory(
    recs,
    switch_at=0,
    z_clip=6.0,
    map_method="sigmoid",
    write_meta=True
)
_dump_records(norm_out_path, norm_recs)  # 기존 dump 그대로 사용
print("[norm] stats:", json.dumps(info, ensure_ascii=False))

[norm] stats: {"median": 0.0, "scale": 0.37065, "scale_source": "mad", "z_clip": 6.0, "map": "sigmoid", "switch_at": 0, "n_tail_values": 481472}


In [3]:
# -----------------------------
# Normalization (PCMI tail only)
# -----------------------------
from dataclasses import dataclass
import copy
import numpy as np
import math
from typing import List, Dict, Any, Optional, Tuple

def _safe_clip01(x: float, eps: float = 1e-6) -> float:
    return float(min(max(x, eps), 1.0 - eps))

def _logit(p: float, eps: float = 1e-6) -> float:
    p = _safe_clip01(p, eps)
    return math.log(p / (1.0 - p))

def _split_pcmi(arr: List[float], switch_at: int) -> Tuple[np.ndarray, np.ndarray]:
    """
    switch_at 기준으로 앞(MC: [0,1]) / 뒤(PCMI raw)를 분리.
    - 값의 범위로 구분하지 말고, merge 정책(인덱스)을 신뢰해 인덱스로 자른다.
    """
    if not isinstance(arr, list) or len(arr) == 0:
        return np.zeros((0,), dtype=np.float64), np.zeros((0,), dtype=np.float64)
    head = np.asarray(arr[:switch_at], dtype=np.float64)
    tail = np.asarray(arr[switch_at:], dtype=np.float64)
    return head, tail

def _fit_sigmoid_quantile_match(
    pmi_vals: np.ndarray,
    mc_vals: np.ndarray,
    q_low: float = 0.10,
    q_high: float = 0.90,
    eps: float = 1e-3,
) -> Tuple[float, float]:
    """
    solve a, b s.t.
        sigmoid(a*(pmi_ql - b)) ~= mc_ql
        sigmoid(a*(pmi_qh - b)) ~= mc_qh
    ==> let L1=logit(mc_ql), L2=logit(mc_qh):
        a = (L2-L1) / (pmi_qh - pmi_ql),  b = pmi_ql - L1/a
    """
    # 안전장치
    if pmi_vals.size == 0:
        return 1.0, float(np.median(mc_vals) if mc_vals.size else 0.0)
    if mc_vals.size == 0:
        # MC가 비어있으면 확률 앵커가 없으므로 보수적으로 약한 스케일
        return 1.0, float(np.median(pmi_vals))

    pmi_ql, pmi_qh = np.quantile(pmi_vals, [q_low, q_high])
    mc_ql,  mc_qh  = np.quantile(mc_vals,  [q_low, q_high])

    # 경계/특이 케이스 보정
    denom = float(pmi_qh - pmi_ql)
    if abs(denom) < 1e-12:
        # PMI 분포 폭이 거의 0 → 작은 스케일만 주고 중앙 정렬
        return 1.0, float(pmi_ql)

    L1 = _logit(float(mc_ql), eps=eps)
    L2 = _logit(float(mc_qh), eps=eps)

    a = (L2 - L1) / denom
    # a 가 너무 작거나 큰 경우 안정화
    if not np.isfinite(a) or abs(a) < 1e-8:
        a = 1.0
    b = pmi_ql - (L1 / a)
    if not np.isfinite(b):
        b = float(pmi_ql)

    return float(a), float(b)

@dataclass
class PCMIParams:
    a: float
    b: float
    switch_at: int
    q_low: float
    q_high: float
    eps: float = 1e-3

def fit_pcmi_params(
    entries: List[dict],
    reward_key: str,
    *,
    switch_at: int,
    q_low: float = 0.10,
    q_high: float = 0.90,
    eps: float = 1e-3,
) -> PCMIParams:
    mc_all, pmi_all = [], []
    for e in entries:
        vec = e.get(reward_key)
        if not isinstance(vec, list):
            continue
        mc, pmi = _split_pcmi(vec, switch_at)
        if mc.size:  mc_all.append(mc)
        if pmi.size: pmi_all.append(pmi)
    mc_all  = np.concatenate(mc_all)  if mc_all  else np.zeros((0,), dtype=np.float64)
    pmi_all = np.concatenate(pmi_all) if pmi_all else np.zeros((0,), dtype=np.float64)

    a, b = _fit_sigmoid_quantile_match(pmi_all, mc_all, q_low=q_low, q_high=q_high, eps=eps)
    return PCMIParams(a=a, b=b, switch_at=int(switch_at), q_low=q_low, q_high=q_high, eps=eps)

def apply_pcmi_calibration(
    entries: List[dict],
    reward_key: str,
    pcmi: PCMIParams,
    *,
    keep_raw: bool = True,
    out_key: Optional[str] = None,
    z_clip: float = 50.0,   # numerical safety
) -> List[dict]:
    """
    MC(앞)는 그대로, PCMI(뒤)만 sigmoid(a*(x-b))로 확률 스케일.
    결과는 (0,1)로 클리핑 → BCEWithLogitsLoss에 바로 사용.
    """
    out_key = out_key or reward_key
    outs: List[dict] = []
    for e in entries:
        vec = e.get(reward_key)
        if not isinstance(vec, list):
            continue
        e2 = copy.deepcopy(e)
        if keep_raw and f"raw_{reward_key}" not in e2:
            e2[f"raw_{reward_key}"] = list(vec)

        mc, pmi = _split_pcmi(vec, pcmi.switch_at)
        new_tail = []
        for v in pmi.tolist():
            z = pcmi.a * (float(v) - pcmi.b)
            # 안정성
            if z_clip is not None:
                z = max(min(z, z_clip), -z_clip)
            prob = 1.0 / (1.0 + math.exp(-z))
            new_tail.append(_safe_clip01(prob, eps=1e-6))

        mapped = mc.tolist() + new_tail
        e2[out_key] = [float(x) for x in mapped]
        # 메타 기록(선택)
        meta = dict(e2.get("meta", {}))
        meta.update({
            "pcmi_calibration": {
                "a": pcmi.a, "b": pcmi.b,
                "switch_at": pcmi.switch_at,
                "q_low": pcmi.q_low, "q_high": pcmi.q_high,
                "type": "quantile_sigmoid"
            }
        })
        e2["meta"] = meta

        outs.append(e2)
    return outs


In [11]:
def main():
    reward_type = "pll_marginal_cont"
    fuzzy_threshold = 0.95
    on_length = "strict"
    switch_at = 0

    base_path = "/home/leena/rs_prm/datasets/hard_80k/qval/ms_qval_qw3_4b.json"
    mi_path = "/home/leena/rs_prm/datasets/hard_80k/cpmi_contrast/ms_cpmi_contr_qw3_4b.json"

    # 1) merge (그대로)
    recs = merge_datasets(
        base_path=base_path,
        mi_path=mi_path,
        reward_type=reward_type,
        on_length=on_length,
        fuzzy_threshold=fuzzy_threshold,
        switch_at=switch_at,
    )

    # 2) tail(PCMI)만 확률 스케일로 보정
    pcmi_params = fit_pcmi_params(recs, reward_key="step_reward", switch_at=switch_at, q_low=0.01, q_high=0.99, eps=1e-3)
    recs_cal = apply_pcmi_calibration(recs, reward_key="step_reward", pcmi=pcmi_params, keep_raw=True)

    # 3) 저장 (BCEWithLogitsLoss로 바로 학습 가능)
    out_path = f"/home/leena/rs_prm/datasets/hard_80k/cpmi_contrast/{switch_at}_qval_{reward_type}_qw3_4b_cal.json"
    _dump_records(out_path, recs_cal)

In [4]:
out = "/home/leena/rs_prm/datasets/hard_80k/pav/ms_pav_qw3_4b.json"
recs = _load_records(out)
pcmi_params = fit_pcmi_params(recs, reward_key="base_reward", switch_at=0, q_low=0.01, q_high=0.99, eps=1e-3)
recs_cal = apply_pcmi_calibration(recs, reward_key="base_reward", pcmi=pcmi_params, keep_raw=True)

out_path = f"/home/leena/rs_prm/datasets/hard_80k/pav/ms_pav_qw3_4b_cal.json"
_dump_records(out_path, recs_cal)

In [12]:
if __name__ == "__main__":
    main()

[merge] total_base=80000 matched_exact=80000 matched_fuzzy=0 missing=0 length_trimmed=0


## Random rewards

In [ ]:
import random

def _rand_unit(dist="uniform", a=1.0, b=1.0) -> float:
    if dist == "uniform":
        return random.random()
    elif dist == "beta":
        # 파라미터 안정성 보장
        a = max(float(a), 1e-8)
        b = max(float(b), 1e-8)
        # 표준 라이브러리엔 beta 난수 없음 -> 간단한 대체(감마 이용) 없이 uniform만 쓰려면 위 라인만 유지
        # 여기서는 random.betavariate 사용 (표준 라이브러리)
        return random.betavariate(a, b)
    else:
        raise ValueError(f"unknown dist: {dist}")

def make_random_tail_from_merged(
    recs: List[Dict[str, Any]],
    switch_at: int,
    seed: Optional[int] = None,
    dist: str = "uniform",   # "uniform" | "beta"
    beta_a: float = 1.0,
    beta_b: float = 1.0,
    write_meta: bool = True
) -> List[Dict[str, Any]]:
    """
    이미 merge된 recs에서 tail(i >= switch_at) 보상들을 0~1 랜덤으로 교체.
    """
    if seed is not None:
        random.seed(seed)

    out = []
    for r in recs:
        rw = r.get("step_reward", [])
        if not isinstance(rw, list) or len(rw) == 0:
            out.append(r); continue
        n = len(rw)
        take = min(switch_at, n)
        new_rw = list(rw[:take])
        # tail 길이
        tail_len = n - take
        if tail_len > 0:
            if dist == "beta":
                tail = [_rand_unit("beta", beta_a, beta_b) for _ in range(tail_len)]
            else:
                tail = [_rand_unit("uniform") for _ in range(tail_len)]
            new_rw.extend(tail)

        new_rec = dict(r)
        new_rec["step_reward"] = new_rw
        # 메타 표기
        if write_meta:
            meta = dict(new_rec.get("meta", {}))
            meta.update({
                "random_tail": {
                    "policy": f"{switch_at}_qval_then_random",
                    "switch_at": switch_at,
                    "dist": dist,
                    "beta_a": beta_a,
                    "beta_b": beta_b,
                    "seed": seed
                }
            })
            new_rec["meta"] = meta
        # reward_type도 명확히 표시
        new_rec["reward_type"] = f"{switch_at}_qval_then_random"
        out.append(new_rec)
    return out

def merge_datasets_with_random_tail(
    base_path: str,
    mi_path: str,
    on_length: str = "strict",
    fuzzy_threshold: float = 0.95,
    switch_at: int = 1,
    seed: Optional[int] = None,
    dist: str = "uniform",  # "uniform" | "beta"
    beta_a: float = 1.0,
    beta_b: float = 1.0
) -> List[Dict[str, Any]]:
    """
    base+mi를 매칭/트림 규칙 그대로 적용하되, tail은 MI대신 랜덤[0,1]로 채운 머지 결과를 생성.
    """
    if seed is not None:
        random.seed(seed)

    base = _load_records(base_path)
    mi   = _load_records(mi_path)

    mi_exact_idx = _build_exact_index(mi)
    mi_q_buckets = _build_question_buckets(mi)

    merged: List[Dict[str, Any]] = []
    n_exact = n_fuzzy = n_missing = n_len_warn = 0

    for b in base:
        key = _make_key(b)
        m = mi_exact_idx.get(key)
        match_mode = "exact"

        if m is None:
            q = _norm_txt(b.get("question", ""))
            candidates = mi_q_buckets.get(q, [])
            if candidates:
                b_steps = _norm_steps(b.get("completion", []))
                best, best_score = None, -1.0
                for cand in candidates:
                    score = _sequence_similarity(b_steps, _norm_steps(cand.get("completion", [])))
                    if score > best_score:
                        best, best_score = cand, score
                if best and best_score >= fuzzy_threshold:
                    m = best
                    match_mode = f"fuzzy({best_score:.3f})"

        steps = b.get("completion", [])
        steps = steps if isinstance(steps, list) else []
        steps_len = len(steps)
        base_rewards = _get_list(b, "base_reward")
        base_len = len(base_rewards)

        # mi 길이는 길이 산정에만 사용 (없으면 steps_len 사용)
        mi_len = 0
        if m is not None:
            # 어떤 mi_*를 쓰든 대개 동일 길이지만, 안전하게 가장 긴 mi_* 길이를 사용
            mi_len = max([len(v) for k, v in m.items() if k.startswith("mi_") and isinstance(v, list)] + [0])

        if on_length == "strict":
            if m is None:
                # strict인데 mi 매칭 실패 -> 스킵(원 규칙과 일치)
                n_missing += 1
                continue
            if not (steps_len == base_len == mi_len and steps_len > 0):
                n_len_warn += 1
                continue
            n = steps_len
        else:
            # trim 규칙: min(steps, base, mi or steps)
            n = min(steps_len, base_len, mi_len if mi_len > 0 else steps_len)

        if n <= 0:
            continue

        take = min(switch_at, n)
        head = base_rewards[:take]
        tail_len = n - take
        if dist == "beta":
            tail = [ _rand_unit("beta", beta_a, beta_b) for _ in range(tail_len) ]
        else:
            tail = [ _rand_unit("uniform") for _ in range(tail_len) ]
        merged_rw = head + tail

        # correct_mask
        correct_mask = None
        if m is not None and isinstance(m.get("correct_mask"), list):
            correct_mask = m["correct_mask"][:n]
        elif isinstance(b.get("correct_mask"), list):
            correct_mask = b["correct_mask"][:n]

        rec = {
            "question": b.get("question"),
            "completion": steps[:n],
            "step_reward": merged_rw,
            "reward_type": f"{switch_at}_qval_then_random",
            "task": b.get("task") or (m.get("task") if isinstance(m, dict) else None),
            "answer": _pick_answer(b) or (_pick_answer(m) if isinstance(m, dict) else None),
            "meta": {
                "base_reward_key": "base_reward",
                "mi_reward_key": None,
                "policy": f"{switch_at}_qval_then_random",
                "trimmed_to": n,
                "base_steps_len": steps_len,
                "base_reward_len": base_len,
                "mi_reward_len": mi_len,
                "source_match": match_mode,
                "random_tail": {
                    "dist": dist,
                    "beta_a": beta_a,
                    "beta_b": beta_b,
                    "seed": seed
                }
            }
        }
        if correct_mask is not None:
            rec["correct_mask"] = correct_mask

        merged.append(rec)
        if match_mode.startswith("exact"):
            n_exact += 1
        else:
            n_fuzzy += 1

    sys.stderr.write(
        f"[rand-merge] total_base={len(base)} matched_exact={n_exact} "
        f"matched_fuzzy={n_fuzzy} missing={n_missing} length_issues={n_len_warn}\n"
    )
    return merged


In [15]:
def main():
    fuzzy_threshold = 0.95
    on_length = "strict"    # strict | trim
    switch_at = 4

    base_path = "/home/leena/rs_prm/datasets/hard_80k/qval/ms_qval_qw3_4b.json"
    mi_path = "/home/leena/rs_prm/datasets/hard_80k/mi_sum2/ms_mi_qw3_4b.json"
    rand_out_path = f"/home/leena/rs_prm/datasets/hard_80k/qval_mi/ms_{switch_at}_qval_random_qw3_4b.json"

    rand_recs2 = merge_datasets_with_random_tail(
        base_path=base_path,
        mi_path=mi_path,
        on_length=on_length,         # "strict"면 mi 매칭 실패/길이 불일치 샘플은 스킵
        fuzzy_threshold=fuzzy_threshold,
        switch_at=switch_at,
        seed=42,
        dist="uniform"               # 또는 "beta", beta_a=2.0, beta_b=5.0
    )
    _dump_records(rand_out_path, rand_recs2)
    

In [16]:
if __name__ == "__main__":
    main()

[rand-merge] total_base=80000 matched_exact=80000 matched_fuzzy=0 missing=0 length_issues=0


## Switch Transition Ratio

In [2]:
# -----------------------------
# Switch-at mixture statistics
# -----------------------------
def _len_or_zero(x):
    return len(x) if isinstance(x, list) else 0

def compute_switch_mix_stats_from_files(
    base_path: str,
    mi_path: str,
    reward_type: str,
    switch_ats: List[int],
    fuzzy_threshold: float = 0.95
) -> Dict[int, Dict[str, float]]:
    """
    base/mi 원본 파일을 로드해 question+completion으로 매칭(정확/퍼지)한 뒤,
    각 switch_at에 대해 "qval steps", "mi steps"의 개수/비율을 집계.

    반환: {s: {"total_steps": ..., "qval_steps": ..., "mi_steps": ...,
                "qval_ratio": ..., "mi_ratio": ...}}
    """
    reward_type = _normalize_reward_type(reward_type)
    base = _load_records(base_path)
    mi   = _load_records(mi_path)

    mi_exact_idx = _build_exact_index(mi)
    mi_q_buckets = _build_question_buckets(mi)

    # 누적 통계 구조
    stats = {s: {"total_steps": 0, "qval_steps": 0, "mi_steps": 0} for s in switch_ats}

    missed = 0
    for b in base:
        key = _make_key(b)
        m = mi_exact_idx.get(key)
        if m is None:
            q = _norm_txt(b.get("question", ""))
            candidates = mi_q_buckets.get(q, [])
            if candidates:
                b_steps = _norm_steps(b.get("completion", []))
                best, best_score = None, -1.0
                for cand in candidates:
                    score = _sequence_similarity(b_steps, _norm_steps(cand.get("completion", [])))
                    if score > best_score:
                        best, best_score = cand, score
                if best and best_score >= fuzzy_threshold:
                    m = best

        # 매칭 실패: 스킵(이 경우 정확히 몇 스텝이 사용 가능한지 판단 불가)
        if m is None:
            missed += 1
            continue

        # 각 예제에서 실제로 사용 가능한 스텝 수(머지 트림 정책과 동일한 min 규칙 가정)
        steps_len = _len_or_zero(b.get("completion", []))
        base_len  = _len_or_zero(b.get("base_reward", []))
        mi_len    = _len_or_zero(m.get(reward_type, []))
        n = min(steps_len, base_len, mi_len if mi_len > 0 else steps_len)
        if n <= 0:
            continue

        for s in switch_ats:
            q_cnt = min(s, n)                # 앞 s 스텝은 qval
            mi_cnt = max(n - s, 0)           # 나머지는 mi
            stats[s]["total_steps"] += n
            stats[s]["qval_steps"]  += q_cnt
            stats[s]["mi_steps"]    += mi_cnt

    # 비율 계산
    out = {}
    for s, d in stats.items():
        tot = d["total_steps"]
        out[s] = {
            "total_steps": int(tot),
            "qval_steps": int(d["qval_steps"]),
            "mi_steps": int(d["mi_steps"]),
            "qval_ratio": (d["qval_steps"] / tot) if tot else 0.0,
            "mi_ratio": (d["mi_steps"] / tot) if tot else 0.0,
            "unmatched_examples_skipped": missed
        }
    return out

def compute_switch_mix_stats_from_merged(
    recs: List[Dict[str, Any]],
    switch_ats: List[int]
) -> Dict[int, Dict[str, float]]:
    """
    이미 merge된 recs를 이용해 빠르게 통계를 냄.
    주의: 다른 switch_at 가정을 '정확히' 반영하진 못함(특히 MI 미스매치로 잘린 케이스).
    """
    stats = {s: {"total_steps": 0, "qval_steps": 0, "mi_steps": 0} for s in switch_ats}
    for r in recs:
        rewards = r.get("step_reward", [])
        if not isinstance(rewards, list):
            continue
        n = len(rewards)
        if n <= 0:
            continue
        for s in switch_ats:
            q_cnt = min(s, n)
            mi_cnt = max(n - s, 0)
            stats[s]["total_steps"] += n
            stats[s]["qval_steps"]  += q_cnt
            stats[s]["mi_steps"]    += mi_cnt

    out = {}
    for s, d in stats.items():
        tot = d["total_steps"]
        out[s] = {
            "total_steps": int(tot),
            "qval_steps": int(d["qval_steps"]),
            "mi_steps": int(d["mi_steps"]),
            "qval_ratio": (d["qval_steps"] / tot) if tot else 0.0,
            "mi_ratio": (d["mi_steps"] / tot) if tot else 0.0
        }
    return out

def print_switch_mix_table(stats: Dict[int, Dict[str, float]]) -> None:
    """
    보기 좋은 표 형태로 프린트
    """
    keys = sorted(stats.keys())
    print(f"{'switch_at':>9} | {'total':>10} | {'qval(#)':>10} | {'mi(#)':>10} | {'qval(%)':>8} | {'mi(%)':>8}")
    print("-" * 68)
    for s in keys:
        d = stats[s]
        tot = d["total_steps"]
        qn  = d["qval_steps"]
        mn  = d["mi_steps"]
        qp  = d["qval_ratio"] * 100.0
        mp  = d["mi_ratio"] * 100.0
        print(f"{s:>9} | {tot:>10} | {qn:>10} | {mn:>10} | {qp:>7.2f}% | {mp:>7.2f}%")
    # unmatched가 있는 경우 표시
    any_unmatched = any("unmatched_examples_skipped" in d for d in stats.values())
    if any_unmatched:
        skipped = stats[keys[0]].get("unmatched_examples_skipped", 0)
        print(f"\n[info] unmatched examples skipped during matching: {skipped}")


In [5]:
if __name__ == "__main__":
    switch_ats = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
    stats = compute_switch_mix_stats_from_files(
        base_path="/home/leena/rs_prm/datasets/hard_80k/qval/ms_qval_qw3_4b.json",
        mi_path="/home/leena/rs_prm/datasets/hard_80k/mi_sum2/ms_mi_qw3_4b.json",
        reward_type="mi_cmi",   # "mi_loo" 등
        switch_ats=switch_ats,
        fuzzy_threshold=0.95
    )
    print_switch_mix_table(stats)

switch_at |      total |    qval(#) |      mi(#) |  qval(%) |    mi(%)
--------------------------------------------------------------------
        0 |     481472 |          0 |     481472 |    0.00% |  100.00%
        1 |     481472 |      80000 |     401472 |   16.62% |   83.38%
        2 |     481472 |     159878 |     321594 |   33.21% |   66.79%
        3 |     481472 |     230869 |     250603 |   47.95% |   52.05%
        4 |     481472 |     287462 |     194010 |   59.70% |   40.30%
        5 |     481472 |     331362 |     150110 |   68.82% |   31.18%
        6 |     481472 |     365092 |     116380 |   75.83% |   24.17%
        7 |     481472 |     391138 |      90334 |   81.24% |   18.76%
        8 |     481472 |     411342 |      70130 |   85.43% |   14.57%
        9 |     481472 |     426988 |      54484 |   88.68% |   11.32%

[info] unmatched examples skipped during matching: 0


# CPMI (Likelihood)

In [4]:
def normalize_mi_tail_rewards_in_memory(
    recs: List[Dict[str, Any]],
    switch_at: int = 1,
    z_clip: float = 6.0,
    map_method: str = "sigmoid",
    write_meta: bool = True
):
    mi_vals = collect_mi_tail_values(recs, switch_at)
    med, scale, scale_src = _robust_scale(mi_vals)

    out = []
    for r in recs:
        rewards = r.get("pll_marginal_cont", [])
        if not isinstance(rewards, list) or len(rewards) == 0:
            out.append(r)
            continue
        new_rewards = list(rewards)
        for i in range(min(len(rewards), switch_at), len(rewards)):
            x = rewards[i]
            if isinstance(x, (int, float)) and math.isfinite(x):
                z = (float(x) - med) / (scale if scale > 0 else 1.0)
                if z_clip is not None and z_clip > 0:
                    z = max(min(z, z_clip), -z_clip)
                new_rewards[i] = _map_to_unit(z, map_method)
        new_rec = dict(r)
        new_rec["step_reward"] = new_rewards
        if write_meta:
            meta = dict(new_rec.get("meta", {}))
            meta.update({
                "mi_tail_normalization": {
                    "type": "robust_z",
                    "median": med,
                    "scale": scale,
                    "scale_source": scale_src,
                    "z_clip": z_clip,
                    "map": map_method,
                    "switch_at": switch_at,
                    "n_tail_values": len(mi_vals)
                }
            })
            new_rec["meta"] = meta
        out.append(new_rec)

    info = {
        "median": med,
        "scale": scale,
        "scale_source": scale_src,
        "z_clip": z_clip,
        "map": map_method,
        "switch_at": switch_at,
        "n_tail_values": len(mi_vals)
    }
    return out, info

recs = _load_records("/home/leena/rs_prm/datasets/hard_80k/cpmi_final/ms_cpmi_fin_test.json")
out_path = f"/home/leena/rs_prm/datasets/hard_80k/cpmi_final/cpmi_fin_test.json"
norm_out_path = out_path.replace(".json", "_norm.json")
norm_recs, info = normalize_mi_tail_rewards_in_memory(
    recs,
    switch_at=0,
    z_clip=6.0,
    map_method="sigmoid",
    write_meta=True
)
_dump_records(norm_out_path, norm_recs)  # 기존 dump 그대로 사용
print("[norm] stats:", json.dumps(info, ensure_ascii=False))

[norm] stats: {"median": 0.0, "scale": 1.0, "scale_source": "fallback:empty", "z_clip": 6.0, "map": "sigmoid", "switch_at": 0, "n_tail_values": 0}


## CMI/LOO/Shapley/Margin

In [ ]:
from typing import Any, Dict, Iterable, List, Optional, Tuple
import torch
import time, math, re, random
from tqdm import tqdm
from datasets import load_dataset
import traceback
from run_profile import RunProfiler  # 네가 쓰던 프로파일러

class MIHardRewardBatch:
    _STEP_RE = re.compile(r"(?:^|\s)(Step\s+\d+\s*:\s*)", flags=re.IGNORECASE)
    _ANS_RE  = re.compile(r"The\s+answer\s+is\s*:\s*(.+?)\s*(?:[+\-]\s*$|\s*$)", flags=re.IGNORECASE | re.DOTALL)

    def __init__(self, model, tokenizer):
        self.model = model
        self.tokenizer = tokenizer
        self.device = next(model.parameters()).device
        self._H_CACHE: Dict[Tuple[str, str, str], float] = {}
        self.prof = RunProfiler()

    def _clean_spaces(self, s: str) -> str:
        s = s.replace("\u200b", " ").replace("\xa0", " ")
        s = re.sub(r"[ \t]+", " ", s)
        s = re.sub(r"\s+\n", "\n", s)
        return s.strip()
    
    def _find_suffix_start(self, full_ids: List[int], tgt_ids: List[int]) -> int:
        if not tgt_ids or len(tgt_ids) > len(full_ids):
            return -1
        Lf, Lt = len(full_ids), len(tgt_ids)
        for start in range(Lf - Lt, -1, -1):
            if full_ids[start:start+Lt] == tgt_ids:
                return start
        return -1

    def build_prompt(self, question: str, tokenizer=None) -> str:
        return f"Problem:\n{question}\nSolution (step-by-step):\n"
    
    def _format_answer_target(self, gold: str) -> str:
        gold = self._clean_spaces(str(gold))
        return f"The answer is: {gold}"
    
    def _ensure_pad_token(self):
        if self.tokenizer.pad_token_id is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
        self.tokenizer.padding_side = "left"

    # ----------------- Math-Shepherd parsing -----------------
    def parse_math_shepherd_record(self, rec: Dict[str, Any]) -> Dict[str, Any]:
        txt = rec.get("label")
        txt = self._clean_spaces(txt)

        # 문제/풀이 분리
        m_first = re.search(r"Step\s+1\s*:", txt, flags=re.IGNORECASE)
        if m_first:
            question = txt[:m_first.start()].strip()
            tail = txt[m_first.start():].strip()
        else:
            question = re.sub(r"The\s+answer\s+is\s*:.*$", "", txt, flags=re.IGNORECASE).strip()
            tail = ""

        # 정답 파싱
        gold_answer = ""
        answer_target = ""
        ma = self._ANS_RE.search(txt)
        if ma:
            body = self._clean_spaces(ma.group(1))
            body = re.sub(r"\s*[+\-]\s*$", "", body)  # 끝의 +/- 정리
            gold_answer = body
            answer_target = self._format_answer_target(body)

        # ---------- NEW: dataset에 붙인 gold_answer가 있으면 무조건 우선 ----------
        ds_gold = rec.get("gold_answer", None)
        if isinstance(ds_gold, str):
            ds_gold = self._clean_spaces(ds_gold)
        if ds_gold:  # gold가 있으면 라벨 기반 추출을 덮어씀
            gold_answer = ds_gold
            answer_target = self._format_answer_target(ds_gold)

        steps: List[str] = []
        step_labels_pm: List[str] = []

        parts = self._STEP_RE.split(tail)
        for idx in range(1, len(parts), 2):
            step_tag = parts[idx]
            after = parts[idx+1] if idx+1 < len(parts) else ""

            # 이번 스텝 본문 경계
            mnext = self._STEP_RE.search(after)
            mans  = re.search(r"The\s+answer\s+is\s*:", after, flags=re.IGNORECASE)
            if mnext and mans:
                next_cut = min(mnext.start(), mans.start())
            elif mnext:
                next_cut = mnext.start()
            elif mans:
                next_cut = mans.start()
            else:
                next_cut = len(after)

            # 본문 후보
            step_text = (step_tag + " " + after[:next_cut]).strip()

            # 1) 스텝 본문 끝에서 우선 +/- 탐지
            pm_in_body = re.search(r"([+\-])\s*$", step_text)
            if pm_in_body:
                pm = pm_in_body.group(1)
                step_text = re.sub(r"[+\-]\s*$", "", step_text).rstrip()
            else:
                # 2) 본문 뒤 suffix의 맨 앞에서 +/- 탐지 (기존 로직)
                suffix = after[next_cut:].lstrip()
                if   suffix.startswith("+"): pm = "+"
                elif suffix.startswith("-"): pm = "-"
                else:                        pm = "+"

            steps.append(step_text)     # <-- 이미 +/− 제거된 클린 스텝
            step_labels_pm.append(pm)

        step_values = rec.get("value")
        if len(step_values) != len(steps):
            L = min(len(step_values), len(steps))
            steps = steps[:L]
            step_values = step_values[:L]
        correct_mask = [1 if pm == "+" else 0 for pm in step_values]
        
        return {
            "question": question,
            "steps": steps,                 # +/− 제거된 본문만
            "step_pm": step_labels_pm,      # 원본 라벨
            "correct_mask": correct_mask,   # +→1, −→0
            "gold_answer": gold_answer,     # ★ 여기 최종 gold (우선순위: rec.gold → 라벨 추출)
            "answer_target": answer_target, # "The answer is: <gold>"
            "task": rec.get("task", None),
            "raw": txt,
        }

    # === Answer log-probabilities (teacher-forced) ==========================
    @torch.no_grad()
    def _logprob_total(self, prompt: str, target: str) -> Tuple[float, float, int]:
        """
        Return (LP_sum, LP_mean, eff_len) where LP_sum = sum_t log p_theta(target_t | prompt + target_<t>)
        This is teacher-forced log-prob of the target suffix given the prompt.
        """
        t0 = time.perf_counter()
        full = prompt + target
        full_enc = self.tokenizer(full, return_tensors="pt", add_special_tokens=True).to(self.device)
        tgt_ids  = self.tokenizer(target, add_special_tokens=False)["input_ids"]
        input_ids = full_enc["input_ids"]
        full_ids = input_ids[0].tolist()
        L_full, L_tgt = len(full_ids), len(tgt_ids)

        if L_tgt == 0:
            return 0.0, 0.0, 0

        # Align the target suffix inside the full sequence
        Lp = self._find_suffix_start(full_ids, tgt_ids)
        if Lp < 0:
            Lp = L_full - L_tgt

        logits = self.model(**full_enc).logits.float()   # [1, L, V]
        logits_shifted = logits[:, :-1, :]               # [1, L-1, V]
        Lm1 = logits_shifted.shape[1]

        start = max(Lp - 1, 0)                           # first token predicting target[0]
        end   = min(start + L_tgt, Lm1)                  # exclusive
        eff_len = end - start
        if eff_len <= 0:
            return 0.0, 0.0, 0

        # gather log p at the gold target tokens
        lp = torch.log_softmax(logits_shifted[0, start:end, :], dim=-1)    # [eff_len, V]
        tgt_tensor = torch.tensor(tgt_ids[:eff_len], device=lp.device, dtype=torch.long)
        lp_gold = lp.gather(dim=-1, index=tgt_tensor.view(-1, 1)).squeeze(-1)  # [eff_len]
        LP_sum = float(lp_gold.sum().item())
        LP_mean = float(LP_sum / eff_len)
        try:
            self.prof.log(tag="mi:lp:call", wall_s=time.perf_counter()-t0,
                        gen_tokens=eff_len, prompt_len=L_full, target_len=L_tgt)
        except Exception:
            pass
        return LP_sum, LP_mean, eff_len

    @torch.no_grad()
    def _logprob_total_batch(self, prompts: List[str], targets: List[str], batch_size: int = 16) -> List[float]:
        """
        Batched LP_sum list for each (prompt, target).
        """
        assert len(prompts) == len(targets)
        self._ensure_pad_token()
        self.model.eval()
        if not hasattr(self, "_H_CACHE"):
            self._H_CACHE = {}

        out = [None] * len(prompts)
        miss_idx, miss_prompts, miss_targets = [], [], []

        for i, (p, t) in enumerate(zip(prompts, targets)):
            key = ("LP|sum", p, "\u241E", t)
            if key in self._H_CACHE:
                out[i] = self._H_CACHE[key]
            else:
                miss_idx.append(i); miss_prompts.append(p); miss_targets.append(t)

        if not miss_idx:
            return out

        use_amp = torch.cuda.is_available()

        for s in range(0, len(miss_idx), batch_size):
            e = min(s + batch_size, len(miss_idx))
            Ps = miss_prompts[s:e]; Ts = miss_targets[s:e]

            with torch.cuda.amp.autocast(dtype=torch.float16, enabled=use_amp):
                full_enc = self.tokenizer(
                    [p + t for p, t in zip(Ps, Ts)],
                    return_tensors="pt", add_special_tokens=True, padding=True, truncation=False
                ).to(self.device)
                logits = self.model(**full_enc).logits.float()    # [B,L,V]
                logits_shifted = logits[:, :-1, :]                # [B,L-1,V]

            # tokenize targets (no specials) for suffix alignment + gather indices
            enc_t = self.tokenizer(Ts, return_tensors="pt", add_special_tokens=False, padding=True, truncation=False)

            B, Lm1, V = logits_shifted.shape
            full_ids = full_enc["input_ids"]
            ids_t = enc_t["input_ids"]

            for bi in range(B):
                tgt_ids_list = ids_t[bi].tolist()
                L_tgt = int((ids_t[bi] != self.tokenizer.pad_token_id).sum().item()) if self.tokenizer.pad_token_id is not None else len(tgt_ids_list)
                if L_tgt <= 0 or Lm1 <= 0:
                    LP_sum = 0.0
                else:
                    tgt_ids = tgt_ids_list[:L_tgt]
                    full_ids_list = full_ids[bi].tolist()
                    Lp = self._find_suffix_start(full_ids_list, tgt_ids)
                    if Lp < 0:  # fallback
                        enc_p_nospec = self.tokenizer(Ps[bi], add_special_tokens=False)
                        Lp = max(len(enc_p_nospec["input_ids"]), 1)

                    start = max(Lp - 1, 0)
                    end   = min(start + L_tgt, Lm1)
                    eff_len = end - start
                    if eff_len <= 0:
                        LP_sum = 0.0
                    else:
                        lp = torch.log_softmax(logits_shifted[bi, start:end, :], dim=-1)   # [eff_len,V]
                        tgt_tensor = torch.tensor(tgt_ids[:eff_len], device=lp.device, dtype=torch.long)
                        lp_gold = lp.gather(dim=-1, index=tgt_tensor.view(-1, 1)).squeeze(-1)
                        LP_sum = float(lp_gold.sum().item())

                gi = miss_idx[s + bi]
                key = ("LP|sum", prompts[gi], "\u241E", targets[gi])
                self._H_CACHE[key] = LP_sum
                out[gi] = LP_sum
        return out
    
    @torch.no_grad()
    def _logprob_mean_batch(self, prompts: List[str], targets: List[str], batch_size: int = 16) -> List[float]:
        """Return list of LP_mean (= per-target-token average log-prob)."""
        sums = self._logprob_total_batch(prompts, targets, batch_size=batch_size)
        # We cached eff_len in entropy path; for PMI 쪽은 길이=tokenizer(target).len (pad 제외)
        means = []
        for p, t, sm in zip(prompts, targets, sums):
            enc_t = self.tokenizer(t, add_special_tokens=False)
            eff_len = len(enc_t["input_ids"])
            means.append(sm / eff_len if eff_len > 0 else 0.0)
        return means

    def _LP_cached_sum(self, prompt: str, target: str):
        if not hasattr(self, "_H_CACHE"):
            self._H_CACHE = {}
        k = ("LP|sum", prompt, "\u241E", target)
        if k in self._H_CACHE:
            return self._H_CACHE[k]
        s, m, L = self._logprob_total(prompt, target)
        self._H_CACHE[k] = s
        self._H_CACHE[("LP|mean", prompt, "\u241E", target)] = m
        self._H_CACHE[("LP|len",  prompt, "\u241E", target)] = L
        return s

    def _LP_cached_mean(self, prompt: str, target: str):
        if not hasattr(self, "_H_CACHE"):
            self._H_CACHE = {}
        k = ("LP|mean", prompt, "\u241E", target)
        if k in self._H_CACHE:
            return self._H_CACHE[k]
        s, m, L = self._logprob_total(prompt, target)
        self._H_CACHE[k] = m
        self._H_CACHE[("LP|sum",  prompt, "\u241E", target)] = s
        self._H_CACHE[("LP|len",  prompt, "\u241E", target)] = L
        return m
    
    # ----------------- MI algorithms -----------------
    # === NEW: PMI/pointwise-CMI based on answer log-prob =========================
    def compute_step_pmi_cmi(self, question: str, steps: List[str], answer_target: str, tokenizer, normalize: bool = False) -> List[float]:
        """
        Sequential pointwise CMI (PMI-style):
        r_i = log p(A | base + prefix_<=i) - log p(A | base + prefix_<i)
        If normalize=True, use per-token mean log-prob instead of sum.
        """
        t0 = time.perf_counter()
        question = re.sub(r' +', ' ', question)
        base = self.build_prompt(question, tokenizer=tokenizer) + "\n"

        vals: List[float] = []
        prompt = base
        LP_prev = self._LP_cached_mean(prompt, answer_target) if normalize \
                else self._LP_cached_sum(prompt, answer_target)
        for s in steps:
            prompt_with = prompt + s.rstrip().rstrip("\n") + "\n"
            LP_with = self._LP_cached_mean(prompt_with, answer_target) if normalize \
                    else self._LP_cached_sum(prompt_with, answer_target)
            vals.append(LP_with - LP_prev)     # <-- PMI difference
            prompt = prompt_with
            LP_prev = LP_with
        try:
            self.prof.log(tag="pmi:cmi:call", num_prompts=len(steps)+1, n=1,
                        wall_s=time.perf_counter()-t0, num_steps=len(steps))
        except Exception:
            pass
        return vals

    def compute_step_pmi_cmi_batch(self, question: str, steps: List[str], answer_target: str, tokenizer, batch_size: int = 16, normalize: bool = False):
        question = re.sub(r' +', ' ', question)
        base = self.build_prompt(question, tokenizer=tokenizer) + "\n"
        prefixes = [base]
        prompt = base
        for s in steps:
            prompt = prompt + s.rstrip().rstrip("\n") + "\n"
            prefixes.append(prompt)
        if normalize:
            LP_prev_list = self._logprob_mean_batch(prefixes[:-1], [answer_target]*len(steps), batch_size=batch_size)
            LP_with_list = self._logprob_mean_batch(prefixes[1:],  [answer_target]*len(steps), batch_size=batch_size)
        else:
            LP_prev_list = self._logprob_total_batch(prefixes[:-1], [answer_target]*len(steps), batch_size=batch_size)
            LP_with_list = self._logprob_total_batch(prefixes[1:],  [answer_target]*len(steps), batch_size=batch_size)

        cmi = [LP_with_list[i] - LP_prev_list[i] for i in range(len(steps))]
        return cmi

    # === PMI-LOO ================================================================
    def compute_step_pmi_loo(self, question: str, steps: List[str], answer_target: str, tokenizer, normalize: bool = False) -> List[float]:
        """
        Leave-one-out contribution under PMI: φ_i = LP(all steps) - LP(all steps without i)
        (정답 로그우도 관점에서 '제거 시 성능감소'를 기여도로 봄)
        """
        question = re.sub(r' +', ' ', question)
        base = self.build_prompt(question, tokenizer=tokenizer) + "\n"
        with_all = base + "".join(s.rstrip().rstrip("\n") + "\n" for s in steps)

        LP_all = self._LP_cached_mean(with_all, answer_target) if normalize \
                else self._LP_cached_sum(with_all, answer_target)

        contribs = []
        for i in range(len(steps)):
            without_i = base + "".join(steps[j].rstrip().rstrip("\n") + "\n"
                                    for j in range(len(steps)) if j != i)
            LP_wo = self._LP_cached_mean(without_i, answer_target) if normalize \
                    else self._LP_cached_sum(without_i, answer_target)
            # with_all - without_i  (크면 i가 유익)
            contribs.append(LP_all - LP_wo)
        return contribs

    def compute_step_pmi_loo_batch(self, question: str, steps: List[str], answer_target: str, tokenizer, batch_size: int = 16, normalize: bool = False) -> List[float]:
        question = re.sub(r' +', ' ', question)
        base = self.build_prompt(question, tokenizer=tokenizer) + "\n"
        with_all = base + "".join(s.rstrip().rstrip("\n") + "\n" for s in steps)

        prompts = [with_all]
        targets = [answer_target]
        for i in range(len(steps)):
            without_i = base + "".join(steps[j].rstrip().rstrip("\n") + "\n"
                                    for j in range(len(steps)) if j != i)
            prompts.append(without_i); targets.append(answer_target)

        if normalize:
            L = self._logprob_mean_batch(prompts, targets, batch_size=batch_size)
        else:
            L = self._logprob_total_batch(prompts, targets, batch_size=batch_size)

        LP_all = L[0]
        return [LP_all - L[i+1] for i in range(len(steps))]

    # === PMI-Marginal ===========================================================
    def compute_step_pmi_marginal(self, question: str, steps: List[str], answer_target: str, tokenizer, normalize: bool = False) -> List[float]:
        """
        Marginal effect of a single step on top of base only: m_i = LP(base + S_i) - LP(base) (순서 의존, 간단 비교용)
        """
        question = re.sub(r' +', ' ', question)
        base = self.build_prompt(question, tokenizer=tokenizer) + "\n"

        LP_base = self._LP_cached_mean(base, answer_target) if normalize \
                else self._LP_cached_sum(base, answer_target)

        out = []
        for s in steps:
            with_i = base + s.rstrip().rstrip("\n") + "\n"
            LP_with = self._LP_cached_mean(with_i, answer_target) if normalize \
                    else self._LP_cached_sum(with_i, answer_target)
            out.append(LP_with - LP_base)
        return out

    def compute_step_pmi_marginal_batch(self, question: str, steps: List[str], answer_target: str, tokenizer, batch_size: int = 16, normalize: bool = False) -> List[float]:
        question = re.sub(r' +', ' ', question)
        base = self.build_prompt(question, tokenizer=tokenizer) + "\n"

        prompts = [base]
        targets = [answer_target]
        for s in steps:
            with_i = base + s.rstrip().rstrip("\n") + "\n"
            prompts.append(with_i); targets.append(answer_target)

        if normalize:
            L = self._logprob_mean_batch(prompts, targets, batch_size=batch_size)
        else:
            L = self._logprob_total_batch(prompts, targets, batch_size=batch_size)

        LP_base = L[0]
        return [L[i+1] - LP_base for i in range(len(steps))]

    # === PMI-Shapley (순열 평균) ================================================
    def compute_step_pmi_shapley(self, question: str, steps: List[str], answer_target: str, tokenizer, n_perm: int = 10, seed: int = 42, normalize: bool = False) -> List[float]:
        """
        Shapley approximation under PMI: φ_i ≈ E_π[ LP(base + prefix_before_i ∪ {i}) - LP(base + prefix_before_i) ]
        """
        question = re.sub(r' +', ' ', question)
        base = self.build_prompt(question, tokenizer=tokenizer) + "\n"
        N = len(steps); rng = random.Random(seed)
        shap = [0.0] * N

        for _ in range(n_perm):
            idxs = list(range(N)); rng.shuffle(idxs)
            prompt = base
            LP_prev = self._LP_cached_mean(prompt, answer_target) if normalize \
                    else self._LP_cached_sum(prompt, answer_target)
            for idx in idxs:
                prompt_with = prompt + steps[idx].rstrip().rstrip("\n") + "\n"
                LP_with = self._LP_cached_mean(prompt_with, answer_target) if normalize \
                        else self._LP_cached_sum(prompt_with, answer_target)
                shap[idx] += (LP_with - LP_prev)
                prompt, LP_prev = prompt_with, LP_with
        return [v / n_perm for v in shap]

    def compute_step_pmi_shapley_batch(self, question: str, steps: List[str], answer_target: str,  tokenizer, n_perm: int = 10, seed: int = 42, batch_size: int = 16, normalize: bool = False) -> List[float]:
        question = re.sub(r' +', ' ', question)
        base = self.build_prompt(question, tokenizer=tokenizer) + "\n"
        N = len(steps); rng = random.Random(seed)

        prompts_before, prompts_after = [], []
        perms = []
        for _ in range(n_perm):
            idxs = list(range(N)); rng.shuffle(idxs)
            perms.append(idxs)
            prompt = base
            for idx in idxs:
                prompts_before.append(prompt)
                prompt = prompt + steps[idx].rstrip().rstrip("\n") + "\n"
                prompts_after.append(prompt)

        targets = [answer_target] * len(prompts_before)
        if normalize:
            LP_before = self._logprob_mean_batch(prompts_before, targets, batch_size=batch_size)
            LP_after  = self._logprob_mean_batch(prompts_after,  targets, batch_size=batch_size)
        else:
            LP_before = self._logprob_total_batch(prompts_before, targets, batch_size=batch_size)
            LP_after  = self._logprob_total_batch(prompts_after,  targets, batch_size=batch_size)

        shap = [0.0] * N
        k = 0
        for idxs in perms:
            for idx in idxs:
                shap[idx] += (LP_after[k] - LP_before[k])
                k += 1
        return [v / n_perm for v in shap]
    
    # --------- Generation용 보조 프롬프트 ----------
    def _build_gen_prompt(self, question: str, steps_prefix: str) -> str:
        return (
            f"Problem:\n{question}\n"
            f"Given Steps:\n{steps_prefix}\n"
            "Instruction: Only output the final answer in one line like:\n"
            "The answer is: <ANSWER>\n"
        )

    def _parse_generated_answer(self, text: str) -> Optional[str]:
        # 생성 텍스트에서 gold answer 추출(현재 패턴과 호환)
        m = self._ANS_RE.search(self._clean_spaces(text))
        if not m: return None
        body = self._clean_spaces(m.group(1))
        body = re.sub(r"\s*[+\-]\s*$", "", body)
        return body if body else None
    
    @torch.no_grad()
    def _generate_answers(self, prompts: List[str], K: int, max_new_tokens: int = 32, temperature: float = 0.7, top_p: float = 0.95, do_sample: bool = True) -> List[List[str]]:
        """
        각 prompt마다 K개 샘플을 생성해 ['The answer is: ...'] 형태의 원문 텍스트 리스트를 반환.
        """
        outs: List[List[str]] = []
        self.model.eval()
        for prompt in prompts:
            enc = self.tokenizer(prompt, return_tensors="pt").to(self.device)
            gen = self.model.generate(
                **enc,
                max_new_tokens=max_new_tokens,
                do_sample=do_sample,
                temperature=temperature,
                top_p=top_p,
                num_return_sequences=K,
                pad_token_id=self.tokenizer.pad_token_id or self.tokenizer.eos_token_id,
                eos_token_id=self.tokenizer.eos_token_id,
            )
            texts = self.tokenizer.batch_decode(gen, skip_special_tokens=True)
            # num_return_sequences=K이면 texts 길이는 K; 프롬프트+출력이 함께 올 수 있으므로 suffix만 취득
            answers = []
            for t in texts:
                # 프롬프트 이후 생성부분만(가능하면) 떼기
                if t.startswith(prompt):
                    gen_part = t[len(prompt):]
                else:
                    gen_part = t
                answers.append(gen_part)
            outs.append(answers)
        return outs

    # --------- K>1 샘플링 기반 PMI-CMI (sequential) ----------
    def compute_step_pmi_cmi_samples(self, question: str, steps: List[str], tokenizer, K: int = 8, max_new_tokens: int = 32,
                                    temperature: float = 0.7, top_p: float = 0.95, normalize: bool = True) -> List[float]:
        """
        샘플 기대값 근사: r_i ≈ (1/K) ∑_{k=1..K} [ log p(A_k | C, s_i) - log p(A_k | C) ], where A_k ~ p(· | C, s_i)  (After 분포에서 샘플링).
        """
        question = re.sub(r' +', ' ', question)
        base_prefix = ""  # steps_prefix 문자열
        cmi_vals: List[float] = []

        for i, s in enumerate(steps):
            # C = (Q, S_{<i}), After = (Q, S_{≤i})
            steps_prefix_before = base_prefix
            steps_prefix_after  = base_prefix + s.rstrip().rstrip("\n") + "\n"

            prompt_before = self._build_gen_prompt(question, steps_prefix_before)
            prompt_after  = self._build_gen_prompt(question, steps_prefix_after)

            # After 분포에서 K개 생성
            gen_list = self._generate_answers([prompt_after], K=K,
                                            max_new_tokens=max_new_tokens,
                                            temperature=temperature, top_p=top_p)[0]

            diffs = []
            for gen_txt in gen_list:
                gold = self._parse_generated_answer(gen_txt)
                if not gold:
                    continue  # 파싱 실패 샘플은 드롭
                target = self._format_answer_target(gold)

                # teacher-forcing으로 두 컨텍스트 점수
                if normalize:
                    lp_with  = self._LP_cached_mean(prompt_after,  target)
                    lp_prev  = self._LP_cached_mean(prompt_before, target)
                else:
                    lp_with  = self._LP_cached_sum(prompt_after,  target)
                    lp_prev  = self._LP_cached_sum(prompt_before, target)

                diffs.append(lp_with - lp_prev)

            # 샘플 평균(비어있으면 0.0)
            cmi_vals.append( sum(diffs)/len(diffs) if diffs else 0.0 )

            # 다음 i로 진행: base_prefix에 현재 스텝 누적
            base_prefix = steps_prefix_after

        return cmi_vals
    
    # ----------------- public: labeling (stream) -----------------
    def mi_labelling(self, *, ds, n_shapley_perm: int = 10, seed: int = 42, ds_task_tag: Optional[str] = None) -> Iterable[Dict[str, Any]]:
        t_ds0 = time.perf_counter()
        for si, rec in tqdm(enumerate(ds)):
            t0 = time.perf_counter()
            parsed = self.parse_math_shepherd_record(rec)
            question      = parsed["question"]
            steps         = parsed["steps"]
            gold_answer   = parsed["gold_answer"]
            answer_target = self._format_answer_target(gold_answer) if gold_answer else ""
            correct_mask  = parsed["correct_mask"]
            task          = parsed.get("task") or ds_task_tag or "mathshepherd"

            if not question or not steps or not answer_target:
                try:
                    self.prof.log(tag="skip_empty", backend="meta", dataset=task, sample_idx=si)
                except Exception:
                    pass
                continue

            # PMI versions
            # --------- CMI ---------
            try:
                if torch.cuda.is_available():
                    try: torch.cuda.reset_peak_memory_stats()
                    except Exception: pass
                t1 = time.perf_counter()
                pll_cmi = self.compute_step_pmi_cmi_batch(question, steps, answer_target, tokenizer=self.tokenizer, batch_size=16, normalize=True)
                wall = time.perf_counter() - t1
                peak = None
                if torch.cuda.is_available():
                    try: peak = torch.cuda.max_memory_allocated() / (1024**3)
                    except Exception: pass
                # prompts ≈ 2 * len(steps)  (prefix_<i>, prefix_<=i)
                self.prof.log(tag="pmi:cmi:sample", wall_s=wall, peak_mem_gb=peak, dataset=task, sample_idx=si, num_steps=len(steps), num_prompts=2 * len(steps), n=1)
            except Exception as e:
                print("[PMI-CMI batch] exception:", repr(e))
                traceback.print_exc(limit=1)
                try:
                    t1 = time.perf_counter()
                    pll_cmi = self.compute_step_pmi_cmi(question, steps, answer_target, tokenizer=self.tokenizer, normalize=True)
                    wall = time.perf_counter() - t1
                    self.prof.log(tag="pmi:cmi:sample", wall_s=wall, peak_mem_gb=None, dataset=task, sample_idx=si, num_steps=len(steps), num_prompts=2 * len(steps), n=1)
                except Exception as e2:
                    print("[PMI-CMI naive] exception:", repr(e2))
                    traceback.print_exc(limit=1)

            # --------- Marginal ---------
            try:
                if torch.cuda.is_available():
                    try: torch.cuda.reset_peak_memory_stats()
                    except Exception: pass
                t1 = time.perf_counter()
                pll_marginal = self.compute_step_pmi_marginal_batch(question, steps, answer_target, tokenizer=self.tokenizer, batch_size=16, normalize=True)
                wall = time.perf_counter() - t1
                peak = None
                if torch.cuda.is_available():
                    try: peak = torch.cuda.max_memory_allocated() / (1024**3)
                    except Exception: pass
                # prompts ≈ len(steps) + 1 (base + each with_i)
                self.prof.log(tag="pmi:marginal:sample", wall_s=wall, peak_mem_gb=peak, dataset=task, sample_idx=si, num_steps=len(steps), num_prompts=len(steps) + 1, n=1)
            except Exception as e:
                print("[PMI-Marginal batch] exception:", repr(e))
                traceback.print_exc(limit=1)
                try:
                    t1 = time.perf_counter()
                    pll_marginal = self.compute_step_pmi_marginal( question, steps, answer_target, tokenizer=self.tokenizer, normalize=True)
                    wall = time.perf_counter() - t1
                    self.prof.log(tag="pmi:marginal:sample", wall_s=wall, peak_mem_gb=None, dataset=task, sample_idx=si, num_steps=len(steps),num_prompts=len(steps) + 1, n=1)
                except Exception as e2:
                    print("[PMI-Marginal naive] exception:", repr(e2))
                    traceback.print_exc(limit=1)

            # --------- LOO ---------
            try:
                if torch.cuda.is_available():
                    try: torch.cuda.reset_peak_memory_stats()
                    except Exception: pass
                t1 = time.perf_counter()
                pll_loo = self.compute_step_pmi_loo_batch(question, steps, answer_target, tokenizer=self.tokenizer,batch_size=16, normalize=True)
                wall = time.perf_counter() - t1
                peak = None
                if torch.cuda.is_available():
                    try: peak = torch.cuda.max_memory_allocated() / (1024**3)
                    except Exception: pass
                # prompts ≈ len(steps) + 1 (with_all + each without_i)
                self.prof.log(tag="pmi:loo:sample", wall_s=wall, peak_mem_gb=peak, dataset=task, sample_idx=si, num_steps=len(steps), num_prompts=len(steps) + 1, n=1)
            except Exception as e:
                print("[PMI-LOO batch] exception:", repr(e))
                traceback.print_exc(limit=1)
                try:
                    t1 = time.perf_counter()
                    pll_loo = self.compute_step_pmi_loo(question, steps, answer_target, tokenizer=self.tokenizer, normalize=True)
                    wall = time.perf_counter() - t1
                    self.prof.log(tag="pmi:loo:sample", wall_s=wall, peak_mem_gb=None, dataset=task, sample_idx=si, num_steps=len(steps), num_prompts=len(steps) + 1, n=1)
                except Exception as e2:
                    print("[PMI-LOO naive] exception:", repr(e2))
                    traceback.print_exc(limit=1)

            # --------- Shapley ---------
            try:
                if torch.cuda.is_available():
                    try: torch.cuda.reset_peak_memory_stats()
                    except Exception: pass
                t1 = time.perf_counter()
                pll_shapley = self.compute_step_pmi_shapley_batch(question, steps, answer_target, tokenizer=self.tokenizer, n_perm=n_shapley_perm, seed=seed, batch_size=16, normalize=True)
                wall = time.perf_counter() - t1
                peak = None
                if torch.cuda.is_available():
                    try: peak = torch.cuda.max_memory_allocated() / (1024**3)
                    except Exception: pass
                # prompts ≈ n_perm * 2 * len(steps) (before/after)
                self.prof.log(tag="pmi:shapley:sample", wall_s=wall, peak_mem_gb=peak, dataset=task, sample_idx=si, num_steps=len(steps), num_prompts=n_shapley_perm * 2 * len(steps), n=1)
            except Exception as e:
                print("[PMI-Shapley batch] exception:", repr(e))
                traceback.print_exc(limit=1)
                try:
                    t1 = time.perf_counter()
                    pll_shapley = self.compute_step_pmi_shapley(question, steps, answer_target, tokenizer=self.tokenizer,n_perm=n_shapley_perm, seed=seed, normalize=True)
                    wall = time.perf_counter() - t1
                    self.prof.log(tag="pmi:shapley:sample", wall_s=wall, peak_mem_gb=None, dataset=task, sample_idx=si, num_steps=len(steps), num_prompts=n_shapley_perm * 2 * len(steps), n=1)
                except Exception as e2:
                    print("[PMI-Shapley naive] exception:", repr(e2))
                    traceback.print_exc(limit=1)

            entry = {
                "question": question,
                "completion": steps,
                "original_answer": gold_answer,
                "answer_target": answer_target,
                "pll_cmi": pll_cmi,
                "pll_loo": pll_loo,
                "pll_shapley": pll_shapley,
                "pll_marginal": pll_marginal,
                "correct_mask": correct_mask,
                "task": task,
            }
            yield entry

            try:
                self.prof.log(tag="sample_total", dataset=task, sample_idx=si, wall_s=time.perf_counter() - t0)
            except Exception:
                pass

        # final profile log
        try:
            self.prof.log(tag="dataset_total", wall_s=time.perf_counter() - t_ds0)
        except Exception:
            pass 


## Sampling

In [ ]:
from typing import Any, Dict, Iterable, List, Optional, Tuple
import torch
import time, math, re, random
from tqdm import tqdm
from datasets import load_dataset
import traceback
from run_profile import RunProfiler  # 네가 쓰던 프로파일러

class CPMISampleReward:
    _STEP_RE = re.compile(r"(?:^|\s)(Step\s+\d+\s*:\s*)", flags=re.IGNORECASE)
    _ANS_RE  = re.compile(r"The\s+answer\s+is\s*:\s*(.+?)\s*(?:[+\-]\s*$|\s*$)", flags=re.IGNORECASE | re.DOTALL)

    def __init__(self, model, tokenizer):
        self.model = model
        self.tokenizer = tokenizer
        self.device = next(model.parameters()).device
        self._H_CACHE: Dict[Tuple[str, str, str], float] = {}
        self.prof = RunProfiler()

    def _clean_spaces(self, s: str) -> str:
        s = s.replace("\u200b", " ").replace("\xa0", " ")
        s = re.sub(r"[ \t]+", " ", s)
        s = re.sub(r"\s+\n", "\n", s)
        return s.strip()
    
    def _find_suffix_start(self, full_ids: List[int], tgt_ids: List[int]) -> int:
        if not tgt_ids or len(tgt_ids) > len(full_ids):
            return -1
        Lf, Lt = len(full_ids), len(tgt_ids)
        for start in range(Lf - Lt, -1, -1):
            if full_ids[start:start+Lt] == tgt_ids:
                return start
        return -1

    def build_prompt(self, question: str, tokenizer=None) -> str:
        return f"Problem:\n{question}\nSolution (step-by-step):\n"
    
    def _format_answer_target(self, gold: str) -> str:
        gold = self._clean_spaces(str(gold))
        return f"The answer is: {gold}"
    
    def _ensure_pad_token(self):
        if self.tokenizer.pad_token_id is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
        self.tokenizer.padding_side = "left"

    # ----------------- Math-Shepherd parsing -----------------
    def parse_math_shepherd_record(self, rec: Dict[str, Any]) -> Dict[str, Any]:
        txt = rec.get("label")
        txt = self._clean_spaces(txt)

        # 문제/풀이 분리
        m_first = re.search(r"Step\s+1\s*:", txt, flags=re.IGNORECASE)
        if m_first:
            question = txt[:m_first.start()].strip()
            tail = txt[m_first.start():].strip()
        else:
            question = re.sub(r"The\s+answer\s+is\s*:.*$", "", txt, flags=re.IGNORECASE).strip()
            tail = ""

        # 정답 파싱
        gold_answer = ""
        answer_target = ""
        ma = self._ANS_RE.search(txt)
        if ma:
            body = self._clean_spaces(ma.group(1))
            body = re.sub(r"\s*[+\-]\s*$", "", body)  # 끝의 +/- 정리
            gold_answer = body
            answer_target = self._format_answer_target(body)

        # ---------- NEW: dataset에 붙인 gold_answer가 있으면 무조건 우선 ----------
        ds_gold = rec.get("gold_answer", None)
        if isinstance(ds_gold, str):
            ds_gold = self._clean_spaces(ds_gold)
        if ds_gold:  # gold가 있으면 라벨 기반 추출을 덮어씀
            gold_answer = ds_gold
            answer_target = self._format_answer_target(ds_gold)

        steps: List[str] = []
        step_labels_pm: List[str] = []

        parts = self._STEP_RE.split(tail)
        for idx in range(1, len(parts), 2):
            step_tag = parts[idx]
            after = parts[idx+1] if idx+1 < len(parts) else ""

            # 이번 스텝 본문 경계
            mnext = self._STEP_RE.search(after)
            mans  = re.search(r"The\s+answer\s+is\s*:", after, flags=re.IGNORECASE)
            if mnext and mans:
                next_cut = min(mnext.start(), mans.start())
            elif mnext:
                next_cut = mnext.start()
            elif mans:
                next_cut = mans.start()
            else:
                next_cut = len(after)

            # 본문 후보
            step_text = (step_tag + " " + after[:next_cut]).strip()

            # 1) 스텝 본문 끝에서 우선 +/- 탐지
            pm_in_body = re.search(r"([+\-])\s*$", step_text)
            if pm_in_body:
                pm = pm_in_body.group(1)
                step_text = re.sub(r"[+\-]\s*$", "", step_text).rstrip()
            else:
                # 2) 본문 뒤 suffix의 맨 앞에서 +/- 탐지 (기존 로직)
                suffix = after[next_cut:].lstrip()
                if   suffix.startswith("+"): pm = "+"
                elif suffix.startswith("-"): pm = "-"
                else:                        pm = "+"

            steps.append(step_text)     # <-- 이미 +/− 제거된 클린 스텝
            step_labels_pm.append(pm)

        step_values = rec.get("value")
        if len(step_values) != len(steps):
            L = min(len(step_values), len(steps))
            steps = steps[:L]
            step_values = step_values[:L]
        correct_mask = [1 if pm == "+" else 0 for pm in step_values]
        
        return {
            "question": question,
            "steps": steps,                 # +/− 제거된 본문만
            "step_pm": step_labels_pm,      # 원본 라벨
            "correct_mask": correct_mask,   # +→1, −→0
            "gold_answer": gold_answer,     # ★ 여기 최종 gold (우선순위: rec.gold → 라벨 추출)
            "answer_target": answer_target, # "The answer is: <gold>"
            "task": rec.get("task", None),
            "raw": txt,
        }

    # === Answer log-probabilities (teacher-forced) ==========================
    @torch.no_grad()
    def _logprob_total(self, prompt: str, target: str) -> Tuple[float, float, int]:
        """
        Return (LP_sum, LP_mean, eff_len) where LP_sum = sum_t log p_theta(target_t | prompt + target_<t>)
        This is teacher-forced log-prob of the target suffix given the prompt.
        """
        t0 = time.perf_counter()
        full = prompt + target
        full_enc = self.tokenizer(full, return_tensors="pt", add_special_tokens=True).to(self.device)
        tgt_ids  = self.tokenizer(target, add_special_tokens=False)["input_ids"]
        input_ids = full_enc["input_ids"]
        full_ids = input_ids[0].tolist()
        L_full, L_tgt = len(full_ids), len(tgt_ids)

        if L_tgt == 0:
            return 0.0, 0.0, 0

        # Align the target suffix inside the full sequence
        Lp = self._find_suffix_start(full_ids, tgt_ids)
        if Lp < 0:
            Lp = L_full - L_tgt

        logits = self.model(**full_enc).logits.float()   # [1, L, V]
        logits_shifted = logits[:, :-1, :]               # [1, L-1, V]
        Lm1 = logits_shifted.shape[1]

        start = max(Lp - 1, 0)                           # first token predicting target[0]
        end   = min(start + L_tgt, Lm1)                  # exclusive
        eff_len = end - start
        if eff_len <= 0:
            return 0.0, 0.0, 0

        # gather log p at the gold target tokens
        lp = torch.log_softmax(logits_shifted[0, start:end, :], dim=-1)    # [eff_len, V]
        tgt_tensor = torch.tensor(tgt_ids[:eff_len], device=lp.device, dtype=torch.long)
        lp_gold = lp.gather(dim=-1, index=tgt_tensor.view(-1, 1)).squeeze(-1)  # [eff_len]
        LP_sum = float(lp_gold.sum().item())
        LP_mean = float(LP_sum / eff_len)
        try:
            self.prof.log(tag="mi:lp:call", wall_s=time.perf_counter()-t0,
                        gen_tokens=eff_len, prompt_len=L_full, target_len=L_tgt)
        except Exception:
            pass
        return LP_sum, LP_mean, eff_len

    @torch.no_grad()
    def _logprob_total_batch(self, prompts: List[str], targets: List[str], batch_size: int = 16) -> List[float]:
        """
        Batched LP_sum list for each (prompt, target).
        """
        assert len(prompts) == len(targets)
        self._ensure_pad_token()
        self.model.eval()
        if not hasattr(self, "_H_CACHE"):
            self._H_CACHE = {}

        out = [None] * len(prompts)
        miss_idx, miss_prompts, miss_targets = [], [], []

        for i, (p, t) in enumerate(zip(prompts, targets)):
            key = ("LP|sum", p, "\u241E", t)
            if key in self._H_CACHE:
                out[i] = self._H_CACHE[key]
            else:
                miss_idx.append(i); miss_prompts.append(p); miss_targets.append(t)

        if not miss_idx:
            return out

        use_amp = torch.cuda.is_available()

        for s in range(0, len(miss_idx), batch_size):
            e = min(s + batch_size, len(miss_idx))
            Ps = miss_prompts[s:e]; Ts = miss_targets[s:e]

            with torch.cuda.amp.autocast(dtype=torch.float16, enabled=use_amp):
                full_enc = self.tokenizer(
                    [p + t for p, t in zip(Ps, Ts)],
                    return_tensors="pt", add_special_tokens=True, padding=True, truncation=False
                ).to(self.device)
                logits = self.model(**full_enc).logits.float()    # [B,L,V]
                logits_shifted = logits[:, :-1, :]                # [B,L-1,V]

            # tokenize targets (no specials) for suffix alignment + gather indices
            enc_t = self.tokenizer(Ts, return_tensors="pt", add_special_tokens=False, padding=True, truncation=False)

            B, Lm1, V = logits_shifted.shape
            full_ids = full_enc["input_ids"]
            ids_t = enc_t["input_ids"]

            for bi in range(B):
                tgt_ids_list = ids_t[bi].tolist()
                L_tgt = int((ids_t[bi] != self.tokenizer.pad_token_id).sum().item()) if self.tokenizer.pad_token_id is not None else len(tgt_ids_list)
                if L_tgt <= 0 or Lm1 <= 0:
                    LP_sum = 0.0
                else:
                    tgt_ids = tgt_ids_list[:L_tgt]
                    full_ids_list = full_ids[bi].tolist()
                    Lp = self._find_suffix_start(full_ids_list, tgt_ids)
                    if Lp < 0:  # fallback
                        enc_p_nospec = self.tokenizer(Ps[bi], add_special_tokens=False)
                        Lp = max(len(enc_p_nospec["input_ids"]), 1)

                    start = max(Lp - 1, 0)
                    end   = min(start + L_tgt, Lm1)
                    eff_len = end - start
                    if eff_len <= 0:
                        LP_sum = 0.0
                    else:
                        lp = torch.log_softmax(logits_shifted[bi, start:end, :], dim=-1)   # [eff_len,V]
                        tgt_tensor = torch.tensor(tgt_ids[:eff_len], device=lp.device, dtype=torch.long)
                        lp_gold = lp.gather(dim=-1, index=tgt_tensor.view(-1, 1)).squeeze(-1)
                        LP_sum = float(lp_gold.sum().item())

                gi = miss_idx[s + bi]
                key = ("LP|sum", prompts[gi], "\u241E", targets[gi])
                self._H_CACHE[key] = LP_sum
                out[gi] = LP_sum
        return out
    
    @torch.no_grad()
    def _logprob_mean_batch(self, prompts: List[str], targets: List[str], batch_size: int = 16) -> List[float]:
        """Return list of LP_mean (= per-target-token average log-prob)."""
        sums = self._logprob_total_batch(prompts, targets, batch_size=batch_size)
        # We cached eff_len in entropy path; for PMI 쪽은 길이=tokenizer(target).len (pad 제외)
        means = []
        for p, t, sm in zip(prompts, targets, sums):
            enc_t = self.tokenizer(t, add_special_tokens=False)
            eff_len = len(enc_t["input_ids"])
            means.append(sm / eff_len if eff_len > 0 else 0.0)
        return means

    def _LP_cached_sum(self, prompt: str, target: str):
        if not hasattr(self, "_H_CACHE"):
            self._H_CACHE = {}
        k = ("LP|sum", prompt, "\u241E", target)
        if k in self._H_CACHE:
            return self._H_CACHE[k]
        s, m, L = self._logprob_total(prompt, target)
        self._H_CACHE[k] = s
        self._H_CACHE[("LP|mean", prompt, "\u241E", target)] = m
        self._H_CACHE[("LP|len",  prompt, "\u241E", target)] = L
        return s

    def _LP_cached_mean(self, prompt: str, target: str):
        if not hasattr(self, "_H_CACHE"):
            self._H_CACHE = {}
        k = ("LP|mean", prompt, "\u241E", target)
        if k in self._H_CACHE:
            return self._H_CACHE[k]
        s, m, L = self._logprob_total(prompt, target)
        self._H_CACHE[k] = m
        self._H_CACHE[("LP|sum",  prompt, "\u241E", target)] = s
        self._H_CACHE[("LP|len",  prompt, "\u241E", target)] = L
        return m
    
    # ----------------- MI algorithm -----------------
    def _build_gen_prompt(self, question: str, steps_prefix: str) -> str:
        return (
            f"Problem:\n{question}\n"
            f"Given Steps:\n{steps_prefix}\n"
            "Instruction: Only output the final answer in one line like:\n"
            "The answer is: <ANSWER>\n"
        )

    def _parse_generated_answer(self, text: str) -> Optional[str]:
        # 생성 텍스트에서 gold answer 추출(현재 패턴과 호환)
        m = self._ANS_RE.search(self._clean_spaces(text))
        if not m: return None
        body = self._clean_spaces(m.group(1))
        body = re.sub(r"\s*[+\-]\s*$", "", body)
        return body if body else None
    
    @torch.no_grad()
    def _generate_answers(self, prompts: List[str], K: int, max_new_tokens: int = 32, temperature: float = 0.7, top_p: float = 0.95, do_sample: bool = True) -> List[List[str]]:
        """
        각 prompt마다 K개 샘플을 생성해 ['The answer is: ...'] 형태의 원문 텍스트 리스트를 반환.
        """
        outs: List[List[str]] = []
        self.model.eval()
        for prompt in prompts:
            enc = self.tokenizer(prompt, return_tensors="pt").to(self.device)
            gen = self.model.generate(
                **enc,
                max_new_tokens=max_new_tokens,
                do_sample=do_sample,
                temperature=temperature,
                top_p=top_p,
                num_return_sequences=K,
                pad_token_id=self.tokenizer.pad_token_id or self.tokenizer.eos_token_id,
                eos_token_id=self.tokenizer.eos_token_id,
            )
            texts = self.tokenizer.batch_decode(gen, skip_special_tokens=True)
            # num_return_sequences=K이면 texts 길이는 K; 프롬프트+출력이 함께 올 수 있으므로 suffix만 취득
            answers = []
            for t in texts:
                # 프롬프트 이후 생성부분만(가능하면) 떼기
                if t.startswith(prompt):
                    gen_part = t[len(prompt):]
                else:
                    gen_part = t
                answers.append(gen_part)
            outs.append(answers)
        return outs

    # --------- K>1 샘플링 기반 PMI-CMI (sequential) ----------
    def compute_step_pmi_cmi_samples(self, question: str, steps: List[str], tokenizer, K: int = 8, max_new_tokens: int = 32,
                                    temperature: float = 0.7, top_p: float = 0.95, normalize: bool = True) -> List[float]:
        """
        샘플 기대값 근사: r_i ≈ (1/K) ∑_{k=1..K} [ log p(A_k | C, s_i) - log p(A_k | C) ], where A_k ~ p(· | C, s_i)  (After 분포에서 샘플링).
        """
        question = re.sub(r' +', ' ', question)
        base_prefix = ""  # steps_prefix 문자열
        cmi_vals: List[float] = []

        for i, s in enumerate(steps):
            # C = (Q, S_{<i}), After = (Q, S_{≤i})
            steps_prefix_before = base_prefix
            steps_prefix_after  = base_prefix + s.rstrip().rstrip("\n") + "\n"

            prompt_before = self._build_gen_prompt(question, steps_prefix_before)
            prompt_after  = self._build_gen_prompt(question, steps_prefix_after)

            # After 분포에서 K개 생성
            gen_list = self._generate_answers([prompt_after], K=K, max_new_tokens=max_new_tokens, temperature=temperature, top_p=top_p)[0]

            diffs = []
            for gen_txt in gen_list:
                gold = self._parse_generated_answer(gen_txt)
                if not gold:
                    continue  # 파싱 실패 샘플은 드롭
                target = self._format_answer_target(gold)

                # teacher-forcing으로 두 컨텍스트 점수
                if normalize:
                    lp_with  = self._LP_cached_mean(prompt_after,  target)
                    lp_prev  = self._LP_cached_mean(prompt_before, target)
                else:
                    lp_with  = self._LP_cached_sum(prompt_after,  target)
                    lp_prev  = self._LP_cached_sum(prompt_before, target)

                diffs.append(lp_with - lp_prev)

            # 샘플 평균(비어있으면 0.0)
            cmi_vals.append( sum(diffs)/len(diffs) if diffs else 0.0 )

            # 다음 i로 진행: base_prefix에 현재 스텝 누적
            base_prefix = steps_prefix_after

        return cmi_vals
    
    # ----------------- public: labeling (stream) -----------------
    def mi_labelling(self, *, ds, n_shapley_perm: int = 10, seed: int = 42, ds_task_tag: Optional[str] = None) -> Iterable[Dict[str, Any]]:
        t_ds0 = time.perf_counter()
        for si, rec in tqdm(enumerate(ds)):
            t0 = time.perf_counter()
            parsed = self.parse_math_shepherd_record(rec)
            question      = parsed["question"]
            steps         = parsed["steps"]
            gold_answer   = parsed["gold_answer"]
            answer_target = self._format_answer_target(gold_answer) if gold_answer else ""
            correct_mask  = parsed["correct_mask"]
            task          = parsed.get("task") or ds_task_tag or "mathshepherd"

            if not question or not steps or not answer_target:
                try:
                    self.prof.log(tag="skip_empty", backend="meta", dataset=task, sample_idx=si)
                except Exception:
                    pass
                continue

            # --------- CMI ---------
            try:
                if torch.cuda.is_available():
                    try: torch.cuda.reset_peak_memory_stats()
                    except Exception: pass
                t1 = time.perf_counter()
                pcmi_sample = self.compute_step_pmi_cmi_samples(question, steps, answer_target, tokenizer=self.tokenizer, batch_size=16, normalize=True)
                wall = time.perf_counter() - t1
                peak = None
                if torch.cuda.is_available():
                    try: peak = torch.cuda.max_memory_allocated() / (1024**3)
                    except Exception: pass
                # prompts ≈ 2 * len(steps)  (prefix_<i>, prefix_<=i)
                self.prof.log(tag="pmi:cmi:sample", wall_s=wall, peak_mem_gb=peak, dataset=task, sample_idx=si, num_steps=len(steps), num_prompts=2 * len(steps), n=1)
            except Exception as e:
                print("[PMI-CMI batch] exception:", repr(e))
                traceback.print_exc(limit=1)
                try:
                    t1 = time.perf_counter()
                    pcmi_sample = self.compute_step_pmi_cmi_samples(question, steps, answer_target, tokenizer=self.tokenizer, normalize=True)
                    wall = time.perf_counter() - t1
                    self.prof.log(tag="pmi:cmi:sample", wall_s=wall, peak_mem_gb=None, dataset=task, sample_idx=si, num_steps=len(steps), num_prompts=2 * len(steps), n=1)
                except Exception as e2:
                    print("[PMI-CMI naive] exception:", repr(e2))
                    traceback.print_exc(limit=1)

            entry = {
                "question": question,
                "completion": steps,
                "original_answer": gold_answer,
                "answer_target": answer_target,
                "pcmi_sample": pcmi_sample,
                "correct_mask": correct_mask,
                "task": task,
            }
            yield entry

            try:
                self.prof.log(tag="sample_total", dataset=task, sample_idx=si, wall_s=time.perf_counter() - t0)
            except Exception:
                pass

        # final profile log
        try:
            self.prof.log(tag="dataset_total", wall_s=time.perf_counter() - t_ds0)
        except Exception:
            pass 


## Contrastive